# 1. Floor plan Parsing

Parses a set of floor plan images into structured JSON: detects Wall/Window/Door objects, finds the outermost slab contour, and assigns compass directions (N/S/E/W) to each floor's outermost openings.


#### 1.1. YOLO11s-seg + SAHI-based Floor plan object recognition

In [ ]:
import os
import json
import cv2
from sahi.predict import get_sliced_prediction
from sahi.models.ultralytics import UltralyticsDetectionModel


# ============================================================
# Configuration
# ============================================================

IMAGE_DIR = "Drawings/FP/Images"
OUTPUT_JSON_DIR = "Drawings/FP/result_FP"

# Pre-trained model weights are not included in this repository.
# Please specify the path to your own compatible model weights.
MODEL_PATH = "path/to/floorplan_model.pt"

DEVICE = "cuda:0"  # Change to "cpu" if CUDA is unavailable.

CONFIDENCE_THRESHOLD = 0.3
SLICE_HEIGHT = 1280
SLICE_WIDTH = 1280
OVERLAP_HEIGHT_RATIO = 0.2
OVERLAP_WIDTH_RATIO = 0.2
POSTPROCESS_MATCH_THRESHOLD = 0.3

TARGET_CLASSES = {"Wall", "Window", "Door"}


def extract_floor_from_filename(filename):
    """
    Extract the floor number from an image filename.

    Expected filename format:
        <name>_<floor_number>.<extension>

    Example:
        floorplan_3.png -> 3
    """
    try:
        return int(filename.split("_")[-1].split(".")[0])
    except (ValueError, IndexError):
        return None


def process_floorplan_images(
    image_dir,
    model_path,
    output_json_dir,
    device=DEVICE,
):
    """
    Detect architectural elements in floor plan images using
    YOLO-based instance segmentation with SAHI sliced inference.

    Detected Wall, Window, and Door instances are exported as
    JSON annotations for subsequent Multi-view-to-BIM processing.

    Parameters
    ----------
    image_dir : str
        Directory containing floor plan images.

    model_path : str
        Path to compatible YOLO segmentation model weights.

    output_json_dir : str
        Directory where annotation JSON files will be saved.

    device : str
        Inference device, e.g. "cuda:0" or "cpu".
    """

    if not os.path.exists(model_path):
        raise FileNotFoundError(
            "Floor plan model weights were not found.\n"
            "Pre-trained weights are not included in this repository.\n"
            "Please specify the path to your own compatible model "
            "in MODEL_PATH."
        )

    if not os.path.isdir(image_dir):
        raise FileNotFoundError(
            f"Floor plan image directory was not found: {image_dir}"
        )

    detection_model = UltralyticsDetectionModel(
        model_path=model_path,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        device=device,
    )

    os.makedirs(output_json_dir, exist_ok=True)

    image_files = sorted(
        file
        for file in os.listdir(image_dir)
        if file.lower().endswith((".png", ".jpg", ".jpeg"))
    )

    for image_file in image_files:

        floor_number = extract_floor_from_filename(image_file)

        if floor_number is None:
            print(
                f"[SKIP] Floor number could not be extracted: "
                f"{image_file}"
            )
            continue

        print(
            f"[PROCESSING] {image_file} "
            f"(Floor: {floor_number})"
        )

        image_path = os.path.join(image_dir, image_file)

        image = cv2.imread(image_path)

        if image is None:
            print(f"[SKIP] Failed to read image: {image_path}")
            continue

        original_height, original_width = image.shape[:2]

        # Run sliced inference using SAHI.
        sahi_result = get_sliced_prediction(
            image=image_path,
            detection_model=detection_model,
            slice_height=SLICE_HEIGHT,
            slice_width=SLICE_WIDTH,
            overlap_height_ratio=OVERLAP_HEIGHT_RATIO,
            overlap_width_ratio=OVERLAP_WIDTH_RATIO,
            postprocess_match_threshold=POSTPROCESS_MATCH_THRESHOLD,
        )

        sahi_annotations = sahi_result.to_coco_annotations()

        coco_annotations = []
        annotation_id = 1

        for obj in sahi_annotations:

            category_name = obj.get("category_name")

            if category_name not in TARGET_CLASSES:
                continue

            bbox = obj.get("bbox")

            if not bbox or len(bbox) != 4:
                print(
                    f"[WARNING] Invalid bounding box in "
                    f"{image_file}: {bbox}"
                )
                continue

            x_min, y_min, width, height = bbox

            x_center = x_min + width / 2
            y_center = y_min + height / 2

            object_width = max(width, height)
            object_depth = min(width, height)

            # The floor number is temporarily used as the Z-coordinate.
            z_value = floor_number

            # Process each segmentation polygon separately.
            segmentation_data = obj.get("segmentation", [])

            if not segmentation_data:
                segmentation_data = [[]]

            for segmentation in segmentation_data:

                annotation = {
                    "id": annotation_id,
                    "image_id": os.path.splitext(image_file)[0],
                    "category_id": obj["category_id"],
                    "category_name": category_name,
                    "original_center": [
                        x_center,
                        y_center,
                    ],
                    "transformed_center": [
                        x_center,
                        y_center,
                        z_value,
                    ],
                    "floor_number": floor_number,
                    "bbox": bbox,
                    "object_width": object_width,
                    "object_depth": object_depth,
                    "score": obj["score"],
                    "segmentation": [segmentation],
                    "iscrowd": obj.get("iscrowd", 0),
                    "area": obj.get("area", 0),
                }

                coco_annotations.append(annotation)
                annotation_id += 1

        output_data = {
            "images": [
                {
                    "id": os.path.splitext(image_file)[0],
                    "file_name": image_file,
                    "height": original_height,
                    "width": original_width,
                }
            ],
            "annotations": coco_annotations,
        }

        output_json_path = os.path.join(
            output_json_dir,
            f"{os.path.splitext(image_file)[0]}.json",
        )

        with open(
            output_json_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                output_data,
                file,
                indent=4,
                ensure_ascii=False,
            )

        print(
            f"[SAVED] Floor plan annotations: "
            f"{output_json_path}"
        )


process_floorplan_images(
    image_dir=IMAGE_DIR,
    model_path=MODEL_PATH,
    output_json_dir=OUTPUT_JSON_DIR,
)

##### Visualization

Renders the detected Wall/Window/Door segmentations on top of the source image and saves the result to disk.


In [ ]:
import os
import json
import cv2
import numpy as np
import random


def _random_bgr():
    """Generate a random BGR color for visualization."""
    return (
        random.randint(40, 255),
        random.randint(40, 255),
        random.randint(40, 255),
    )


def _normalize_segments(segmentation):
    """
    Normalize segmentation data into a list of polygon coordinate lists.

    Supports both:
    - Flattened format: [x1, y1, x2, y2, ...]
    - Nested format: [[x1, y1, x2, y2, ...], [...]]
    """
    if not segmentation:
        return []

    if (
        isinstance(segmentation, list)
        and segmentation
        and isinstance(segmentation[0], (int, float))
    ):
        return [segmentation]

    return segmentation


def visualize_and_save(
    image_path,
    json_path,
    output_path,
    dark_mode=False,
    use_bbox=False,
    line_thickness=2,
):
    """
    Visualize floor plan detection results using OpenCV and save them as PNG.

    Parameters
    ----------
    image_path : str
        Path to the input floor plan image.

    json_path : str
        Path to the corresponding annotation JSON file.

    output_path : str
        Path where the visualization image will be saved.

    dark_mode : bool
        If True, annotations are drawn on a black background.
        If False, annotations are drawn on the original image.

    use_bbox : bool
        If True, bounding boxes are visualized.
        If False, segmentation polygons are visualized.

    line_thickness : int
        Thickness of the visualization lines.
    """

    image = cv2.imread(image_path)

    if image is None:
        print(f"[SKIP] Image not found: {image_path}")
        return

    # Use either a black background or the original image.
    canvas = np.zeros_like(image) if dark_mode else image.copy()

    with open(json_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    annotations = data.get("annotations", [])

    for annotation in annotations:

        category_id = annotation.get("category_id")
        category_name = annotation.get("category_name", "")

        # Process Wall, Window, and Door annotations only.
        if (
            category_id not in (0, 1, 2)
            and category_name not in ("Wall", "Window", "Door")
        ):
            continue

        color = _random_bgr()

        if use_bbox:
            bbox = annotation.get("bbox", [])

            if bbox and len(bbox) >= 4:
                x, y, width, height = bbox[:4]

                x1 = int(round(x))
                y1 = int(round(y))
                x2 = int(round(x + width))
                y2 = int(round(y + height))

                cv2.rectangle(
                    canvas,
                    (x1, y1),
                    (x2, y2),
                    color,
                    thickness=max(1, line_thickness),
                )

        else:
            segmentation = annotation.get("segmentation", [])

            for segment in _normalize_segments(segmentation):

                coordinates = np.asarray(
                    segment,
                    dtype=np.float32,
                )

                # At least two points are required.
                if coordinates.size < 4:
                    continue

                try:
                    points = coordinates.reshape(-1, 2).astype(np.int32)
                except (ValueError, TypeError):
                    continue

                cv2.polylines(
                    canvas,
                    [points],
                    isClosed=True,
                    color=color,
                    thickness=max(1, line_thickness),
                )

    output_directory = os.path.dirname(output_path)

    if output_directory:
        os.makedirs(output_directory, exist_ok=True)

    saved = cv2.imwrite(output_path, canvas)

    if saved:
        print(f"[SAVED] {output_path}")
    else:
        print(f"[ERROR] Failed to save: {output_path}")


def process_folder(
    images_dir,
    jsons_dir,
    output_dir,
    dark_mode=False,
    use_bbox=False,
    line_thickness=2,
):
    """
    Match floor plan images with corresponding JSON annotation files
    and generate visualization images for all valid pairs.

    Image and JSON files are matched based on their filenames
    excluding the file extension.
    """

    os.makedirs(output_dir, exist_ok=True)

    json_index = {
        os.path.splitext(filename)[0]: os.path.join(jsons_dir, filename)
        for filename in os.listdir(jsons_dir)
        if filename.lower().endswith(".json")
    }

    saved_count = 0

    for filename in sorted(os.listdir(images_dir)):

        if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
            continue

        stem = os.path.splitext(filename)[0]

        image_path = os.path.join(images_dir, filename)
        json_path = json_index.get(stem)

        if json_path is None:
            print(f"[SKIP] No matching JSON found for: {filename}")
            continue

        output_path = os.path.join(
            output_dir,
            f"{stem}.png",
        )

        visualize_and_save(
            image_path=image_path,
            json_path=json_path,
            output_path=output_path,
            dark_mode=dark_mode,
            use_bbox=use_bbox,
            line_thickness=line_thickness,
        )

        saved_count += 1

    print(
        f"[DONE] {saved_count} visualization images "
        f"saved to {output_dir}"
    )


# Generate visualizations for floor plan parsing results.
process_folder(
    images_dir="Drawings/FP/Images",
    jsons_dir="Drawings/FP/result_FP",
    output_dir="Drawings/FP/viz_FP",
    dark_mode=False,
    use_bbox=False,
    line_thickness=2,
)

#### 1.2. Outermost contour detection of recognized objects

In [ ]:
import os
import json
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union


def load_json(json_path):
    """Load a JSON file."""
    with open(json_path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(json_data, output_path):
    """Save JSON data to a file."""
    output_directory = os.path.dirname(output_path)

    if output_directory:
        os.makedirs(output_directory, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(json_data, file, indent=4)

    print(f"[SAVED] Updated JSON: {output_path}")


def segmentation_to_polygons(segmentation):
    """
    Convert segmentation data into Shapely polygons.

    Each segmentation polygon is expected in the form:
        [x1, y1, x2, y2, ...]
    """
    polygons = [
        Polygon(np.asarray(segment).reshape(-1, 2))
        for segment in segmentation
        if len(segment) >= 6
    ]

    return polygons


def polygon_to_segmentation(polygon):
    """
    Convert a Shapely Polygon or MultiPolygon into
    segmentation coordinate lists.
    """
    if isinstance(polygon, MultiPolygon):
        return [
            list(np.asarray(poly.exterior.coords).flatten())
            for poly in polygon.geoms
        ]

    return [
        list(np.asarray(polygon.exterior.coords).flatten())
    ]


def get_connected_components(polygons):
    """
    Find connected groups of intersecting polygons.
    """
    num_polygons = len(polygons)
    visited = [False] * num_polygons
    components = []

    for i in range(num_polygons):

        if not visited[i]:
            stack = [i]
            component = []

            while stack:
                current = stack.pop()

                if not visited[current]:
                    visited[current] = True
                    component.append(current)

                    for j in range(num_polygons):
                        if (
                            not visited[j]
                            and polygons[current].intersects(polygons[j])
                        ):
                            stack.append(j)

            components.append(component)

    return components


def merge_wall_segmentations(json_data, buffer_distance=5):
    """
    Merge spatially connected Wall and Window segmentations.

    Processing steps
    ----------------
    1. Extract Wall and Window annotations.
    2. Convert their segmentation data into Shapely polygons.
    3. Expand each polygon using the specified buffer distance.
    4. Identify connected groups of buffered polygons.
    5. Merge polygons within each connected group.
    6. Replace the original Wall and Window annotations with
       merged Wall annotations.

    Parameters
    ----------
    json_data : dict
        Input annotation data.

    buffer_distance : float
        Buffer distance applied to each polygon before determining
        spatial connectivity.
    """

    wall_window_annotations = []
    other_annotations = []

    for annotation in json_data.get("annotations", []):

        if annotation.get("category_name") in ("Wall", "Window"):
            wall_window_annotations.append(annotation)
        else:
            other_annotations.append(annotation)

    if not wall_window_annotations:
        print("[INFO] No Wall or Window annotations found.")
        return json_data

    wall_polygons = []

    for annotation in wall_window_annotations:
        segmentation = annotation.get("segmentation", [])

        polygons = segmentation_to_polygons(segmentation)
        wall_polygons.extend(polygons)

    if not wall_polygons:
        print("[INFO] No valid Wall or Window polygons found.")
        return json_data

    # Apply a buffer to each polygon before determining connectivity.
    buffered_polygons = [
        polygon.buffer(buffer_distance)
        for polygon in wall_polygons
    ]

    components = get_connected_components(buffered_polygons)

    annotations = json_data.get("annotations", [])

    next_id = (
        max(annotation["id"] for annotation in annotations) + 1
        if annotations
        else 1
    )

    merged_annotations = []

    for component in components:

        component_polygons = [
            buffered_polygons[index]
            for index in component
        ]

        merged_polygon = unary_union(component_polygons)

        merged_segmentation = polygon_to_segmentation(
            merged_polygon
        )

        new_annotation = {
            "id": next_id,
            "image_id": annotations[0]["image_id"] if annotations else None,
            "category_id": 2,
            "category_name": "Wall",
            "segmentation": merged_segmentation,

            # The original research implementation stores
            # Shapely bounds as [min_x, min_y, max_x, max_y].
            "bbox": list(merged_polygon.bounds),

            "area": merged_polygon.area,
        }

        merged_annotations.append(new_annotation)
        next_id += 1

    # Remove the original Wall and Window annotations and replace them
    # with the merged Wall annotations.
    json_data["annotations"] = (
        other_annotations + merged_annotations
    )

    return json_data


def process_wall_merging(
    json_path,
    output_path,
    buffer_distance=5,
):
    """
    Load floor plan annotations, merge connected Wall and Window
    segmentations, and save the updated annotations.
    """
    json_data = load_json(json_path)

    updated_json = merge_wall_segmentations(
        json_data,
        buffer_distance=buffer_distance,
    )

    save_json(updated_json, output_path)


# Merge Wall and Window segmentations for a floor plan.
process_wall_merging(
    json_path="Drawings/FP/result_FP/TEST_1.json",
    output_path="Drawings/FP/result2_FP/TEST_1.json",
    buffer_distance=5,
)

In [ ]:
import os
import json
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

def load_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(json_data, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=4)
    print(f"[SAVED] {output_path}")

def segmentation_to_polygon(segmentation):
    polygons = [Polygon(np.array(seg).reshape(-1, 2)) for seg in segmentation if len(seg) >= 6]
    return polygons

def polygon_to_segmentation(polygon):
    if isinstance(polygon, MultiPolygon):
        return [list(np.array(poly.exterior.coords).flatten()) for poly in polygon.geoms]
    else:
        return [list(np.array(polygon.exterior.coords).flatten())]

def get_connected_components(polygons, buffer_distance=0):
    n = len(polygons)
    visited = [False] * n
    components = []
    for i in range(n):
        if not visited[i]:
            stack = [i]
            comp = []
            while stack:
                cur = stack.pop()
                if not visited[cur]:
                    visited[cur] = True
                    comp.append(cur)
                    for j in range(n):
                        if not visited[j] and polygons[cur].intersects(polygons[j]):
                            stack.append(j)
            components.append(comp)
    return components

def merge_wall_segmentations(json_data, buffer_distance=5):
    wall_annotations = []
    window_annotations = []
    other_annotations = []

    # Split annotations by category
    for ann in json_data.get("annotations", []):
        cname = ann.get("category_name")
        if cname == "Wall":
            wall_annotations.append(ann)
        elif cname == "Window":
            window_annotations.append(ann)
        else:
            other_annotations.append(ann)

    if not wall_annotations and not window_annotations:
        # Nothing to merge
        return json_data

    # Exclude the lowest-ID Window from the union (kept in the output, just not merged)
    window_annotations_sorted = sorted(window_annotations, key=lambda x: x["id"])
    excluded_window = window_annotations_sorted[0] if window_annotations_sorted else None  # unused unless logic changes
    window_union_targets = window_annotations_sorted[1:] if len(window_annotations_sorted) > 1 else []

    # Convert to polygons (Walls + Windows excluding the first one)
    polygons = []
    for ann in wall_annotations + window_union_targets:
        polys = segmentation_to_polygon(ann.get("segmentation", []))
        polygons.extend(polys)

    if not polygons:
        # No valid polygons, return unchanged
        return json_data

    # Buffer each polygon, then find connected components
    buffered_polygons = [poly.buffer(buffer_distance) for poly in polygons]
    components = get_connected_components(buffered_polygons, buffer_distance=0)

    next_id = (max([ann["id"] for ann in json_data.get("annotations", [])]) + 1) if json_data.get("annotations") else 1
    merged_annotations = []

    for comp in components:
        comp_polygons = [buffered_polygons[i] for i in comp]
        merged_polygon = unary_union(comp_polygons)
        merged_segmentation = polygon_to_segmentation(merged_polygon)

        new_annotation = {
            "id": next_id,
            "image_id": json_data["annotations"][0]["image_id"] if json_data.get("annotations") else None,
            "category_id": 2,
            "category_name": "Wall",
            "segmentation": merged_segmentation,
            "bbox": list(merged_polygon.bounds),
            "area": merged_polygon.area
        }
        merged_annotations.append(new_annotation)
        next_id += 1

    # Final: drop the original Walls, keep every Window/other annotation, add the merged Walls
    json_data["annotations"] = (
        [ann for ann in json_data.get("annotations", []) if ann.get("category_name") != "Wall"]
        + merged_annotations
    )
    return json_data

def merge_walls_for_file(json_path, output_path, buffer_distance=5):
    data = load_json(json_path)
    updated = merge_wall_segmentations(data, buffer_distance=buffer_distance)
    save_json(updated, output_path)
    print(f"[DONE] {json_path}")

# ---------- Batch-process a folder (optionally excluding one file) ----------
def process_folder(input_dir, output_dir, buffer_distance=5, recursive=True, exclude_one=None):
    """
    Process every .json file in input_dir and save the results to output_dir.
    - recursive: also walk subfolders
    - exclude_one: a single file to skip; accepts an absolute/relative path,
      or a bare filename (matched by basename)
    """
    exclude_abs = None
    exclude_base = None
    if exclude_one:
        exclude_abs = os.path.abspath(exclude_one)
        exclude_base = os.path.basename(exclude_one)

    tasks = []
    if recursive:
        for root, _, files in os.walk(input_dir):
            for fn in files:
                if fn.lower().endswith(".json"):
                    in_path = os.path.join(root, fn)

                    if exclude_one:
                        if os.path.abspath(in_path) == exclude_abs or os.path.basename(in_path) == exclude_base:
                            print(f"[SKIP] Excluded: {in_path}")
                            continue

                    rel = os.path.relpath(in_path, input_dir)
                    out_path = os.path.join(output_dir, rel)
                    tasks.append((in_path, out_path))
    else:
        for fn in os.listdir(input_dir):
            if fn.lower().endswith(".json"):
                in_path = os.path.join(input_dir, fn)

                if exclude_one:
                    if os.path.abspath(in_path) == exclude_abs or os.path.basename(in_path) == exclude_base:
                        print(f"[SKIP] Excluded: {in_path}")
                        continue

                out_path = os.path.join(output_dir, fn)
                tasks.append((in_path, out_path))

    if not tasks:
        print("[INFO] No files to process.")
        return

    print(f"[INFO] Starting processing of {len(tasks)} files")
    for i, (src, dst) in enumerate(tasks, 1):
        try:
            print(f"[{i}/{len(tasks)}] {src} -> {dst}")
            merge_walls_for_file(src, dst, buffer_distance=buffer_distance)
        except Exception as e:
            print(f"[ERROR] {src}: {e}")


# Batch-process the folder, excluding one file (path or bare filename)
process_folder(
    input_dir="Drawings/FP/result_FP",
    output_dir="Drawings/FP/result2_FP",
    buffer_distance=10,
    recursive=True,
    exclude_one="Drawings/FP/result_FP/TEST_1.json"  # or just "TEST_1.json"
)

In [ ]:
import os
import json
import numpy as np
import cv2

def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def extract_contours(json_data, image_size, min_area=3000):
    """
    Fill a mask from COCO-style segmentation and return the single
    largest outer contour (empty list if none is found).
    """
    W, H = image_size
    mask = np.zeros((H, W), dtype=np.uint8)

    for ann in json_data.get("annotations", []):
        seg = ann.get("segmentation", [])
        if not seg:
            continue

        # Handle both flattened and nested [[...]] segmentation formats
        segments = [seg] if (seg and not isinstance(seg[0], list)) else seg

        for segment in segments:
            arr = np.asarray(segment, dtype=np.float32)
            if arr.size < 4 or (arr.size % 2) != 0:
                continue
            pts = arr.reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(mask, [pts], 255)

    # Morphological cleanup: close small gaps and holes
    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return []
    # Filter by area threshold
    contours = [c for c in contours if cv2.contourArea(c) > min_area]
    if not contours:
        return []
    # Keep only the single largest contour
    return [max(contours, key=cv2.contourArea)]

def save_json(json_data, contours, output_file):
    annotations = json_data.setdefault("annotations", [])
    images = json_data.get("images", [])
    img_info = images[0] if images else {"id": None, "width": None, "height": None}

    next_id = (max([a.get("id", 0) for a in annotations]) + 1) if annotations else 1

    try:
        floor_number = int(annotations[0].get("floor_number", 0)) if annotations else 0
    except Exception:
        floor_number = 0

    # Store the contour as a flat coordinate list
    if contours:
        cnt = contours[0].squeeze()
        if cnt.ndim == 1:
            cnt = cnt.reshape(-1, 2)
        pts = cnt.astype(np.int32)
        slab_flat = [int(c) for p in pts for c in p]
        center = np.mean(pts, axis=0).tolist()
        x_min, y_min = int(np.min(pts[:, 0])), int(np.min(pts[:, 1]))
        x_max, y_max = int(np.max(pts[:, 0])), int(np.max(pts[:, 1]))
        bbox = [x_min, y_min, x_max - x_min, y_max - y_min]
        area = float(cv2.contourArea(pts.astype(np.float32)))
        transformed_center = center + [floor_number]
    else:
        slab_flat = []
        center = [0, 0]
        bbox = [0, 0, 0, 0]
        area = 0.0
        transformed_center = [0, 0, floor_number]

    slab_entry = {
        "id": next_id,
        "image_id": img_info.get("id"),
        "category_id": 3,
        "category_name": "Slab",
        "original_center": center,
        "transformed_center": transformed_center,
        "floor_number": floor_number,
        "bbox": bbox,
        "segmentation": slab_flat,  # flat list
        "iscrowd": 0,
        "area": area
    }

    annotations.append(slab_entry)

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=4, ensure_ascii=False)
    print(f"[SAVED JSON] {output_file}")

def save_contours_image(contours, width, height, save_path, color=(0,0,0), thickness=3, bg=(255,255,255), scale_dpi=96):
    """
    Render and save the visualization with OpenCV only (no Matplotlib).
    - color: line color in (B, G, R)
    - thickness: line width
    - scale_dpi: scale factor relative to 96 DPI (OpenCV has no native DPI concept)
    """
    canvas = np.full((height, width, 3), bg, dtype=np.uint8)

    if contours:
        cnt = contours[0].squeeze()
        if cnt.ndim == 1:
            cnt = cnt.reshape(-1, 2)
        pts = cnt.astype(np.int32)
        cv2.polylines(canvas, [pts], isClosed=True, color=color, thickness=thickness)

    # Emulate DPI by upscaling the image
    scale = max(1.0, float(scale_dpi) / 96.0)
    if scale > 1.0:
        new_w = int(round(width * scale))
        new_h = int(round(height * scale))
        canvas = cv2.resize(canvas, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    cv2.imwrite(save_path, canvas)
    print(f"[SAVED IMG] {save_path}")

def run_once(json_file, output_json, visualize=False, min_area=3000, output_img_path=None, img_line_thickness=3, img_color=(0,0,0), img_bg=(255,255,255), img_dpi=300):
    data = load_json(json_file)
    if data.get("images"):
        W = data["images"][0].get("width")
        H = data["images"][0].get("height")
        if W is None or H is None:
            raise ValueError(f"Missing width/height in images[0] for {json_file}")
    else:
        raise ValueError(f"No 'images' field in {json_file}")

    contours = extract_contours(data, (W, H), min_area=min_area)
    save_json(data, contours, output_json)

    # Saving the visualization to a file is recommended
    if output_img_path:
        save_contours_image(
            contours, W, H, output_img_path,
            color=img_color,              # (0,0,0) = black line
            thickness=img_line_thickness, # line width
            bg=img_bg,                    # white background
            scale_dpi=img_dpi             # DPI emulation (upscale)
        )

    # On-screen display is intentionally skipped (avoids NumPy/Matplotlib conflicts)

def process_folder(input_dir, output_dir, output_img_dir=None, recursive=True, visualize=False, min_area=3000, img_line_thickness=3, img_color=(0,0,0), img_bg=(255,255,255), img_dpi=300):
    """
    Read every .json in input_dir, add a Slab annotation, and save to output_dir.
    If output_img_dir is given, also save a .png visualization at the matching
    relative path. Uses OpenCV only.
    """
    tasks = []
    if recursive:
        for root, _, files in os.walk(input_dir):
            for fn in files:
                if fn.lower().endswith(".json"):
                    in_path = os.path.join(root, fn)
                    rel = os.path.relpath(in_path, input_dir)
                    out_json = os.path.join(output_dir, rel)
                    out_img = None
                    if output_img_dir:
                        rel_png = os.path.splitext(rel)[0] + ".png"
                        out_img = os.path.join(output_img_dir, rel_png)
                    tasks.append((in_path, out_json, out_img))
    else:
        for fn in os.listdir(input_dir):
            if fn.lower().endswith(".json"):
                in_path = os.path.join(input_dir, fn)
                out_json = os.path.join(output_dir, fn)
                out_img = os.path.join(output_img_dir, os.path.splitext(fn)[0] + ".png") if output_img_dir else None
                tasks.append((in_path, out_json, out_img))

    if not tasks:
        print("[INFO] No JSON files to process.")
        return

    print(f"[INFO] Processing {len(tasks)} files...")
    for i, (src, dst_json, dst_img) in enumerate(tasks, 1):
        try:
            print(f"[{i}/{len(tasks)}] {src} -> {dst_json}" + (f" | IMG -> {dst_img}" if dst_img else ""))
            os.makedirs(os.path.dirname(dst_json), exist_ok=True)
            if dst_img:
                os.makedirs(os.path.dirname(dst_img), exist_ok=True)

            run_once(
                src, dst_json,
                visualize=False,
                min_area=min_area,
                output_img_path=dst_img,
                img_line_thickness=img_line_thickness,
                img_color=img_color,
                img_bg=img_bg,
                img_dpi=img_dpi
            )
        except Exception as e:
            print(f"[ERROR] {src}: {e}")

# ===== Example run =====
process_folder(
    input_dir="Drawings/FP/result2_FP",
    output_dir="Drawings/FP/result3_FP",
    output_img_dir="Drawings/FP/viz3_FP",
    recursive=True,
    visualize=False,           # on-screen display disabled (Matplotlib removed)
    min_area=3000,
    img_line_thickness=3,      # line width
    img_color=(0,0,0),         # black line (BGR)
    img_bg=(255,255,255),      # white background
    img_dpi=300                # DPI emulation (upscale factor ~300/96 = 3.125x)
)

#### 1.3.  Directional information assignment to contour lines visible on elevations.

In [ ]:
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
import os

def load_json(file_path):
    """Load a JSON file."""
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

def extract_contours(json_data, image_size, min_area=3000):
    """Extract contours from JSON data and drop small ones."""
    mask = np.zeros((image_size[1], image_size[0]), dtype=np.uint8)

    for ann in json_data["annotations"]:
        seg_data = ann.get("segmentation", [])
        if not seg_data:
            continue
        # Wrap flattened segmentation data in a list
        if not isinstance(seg_data[0], list):
            segments = [seg_data]
        else:
            segments = seg_data
        for seg in segments:
            try:
                points = np.array(seg).reshape(-1, 2)
                points = points.astype(np.int32)
                cv2.fillPoly(mask, [points], 255)
            except Exception:
                # Skip on conversion failure
                continue

    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    filtered_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_area]
    if filtered_contours:
        return max(filtered_contours, key=cv2.contourArea)
    return None

def get_directional_lines(contour):
    """Compute the directional boundary lines and the full Slab outline."""
    points = contour.squeeze()

    north_points = {}
    south_points = {}
    x_values_per_y = defaultdict(list)

    slab_contour = []

    for x, y in points:
        slab_contour.extend([int(x), int(y)])  # accumulate the full Slab outline
        if x not in north_points or y < north_points[x]:
            north_points[x] = y
        if x not in south_points or y > south_points[x]:
            south_points[x] = y
        x_values_per_y[y].append(x)

    west_points = {y: min(x_list) for y, x_list in x_values_per_y.items()}
    east_points = {y: max(x_list) for y, x_list in x_values_per_y.items()}

    return slab_contour, north_points, south_points, west_points, east_points

def save_coordinates_to_json(json_file, slab_contour, north_points, south_points, east_points, west_points):
    """Save the directional lines and full Slab outline to JSON, correctly ordered."""
    base_filename = os.path.splitext(os.path.basename(json_file))[0]
    output_dir = "Drawings/FP/result4_FP"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{base_filename}.json")

    directional_data = {
        "Slab": [int(coord) for coord in slab_contour],  # full Slab outline
        "North": [int(coord) for xy in sorted(north_points.items()) for coord in xy],  # flattened 1D list
        "South": [int(coord) for xy in sorted(south_points.items()) for coord in xy],  # flattened 1D list
        "East": [int(coord) for yx in sorted([(x, y) for y, x in east_points.items()], key=lambda p: p[1]) for coord in yx],  # flattened 1D list
        "West": [int(coord) for yx in sorted([(x, y) for y, x in west_points.items()], key=lambda p: p[1]) for coord in yx]  # flattened 1D list
    }

    with open(output_path, "w") as f:
        json.dump(directional_data, f, indent=4)

    print(f"[SAVED] {output_path}")

def process_directional_lines(json_file):
    """Run the full directional-line extraction for one JSON file."""
    json_data = load_json(json_file)
    image_size = (json_data["images"][0]["width"], json_data["images"][0]["height"])

    contour = extract_contours(json_data, image_size)

    if contour is None:
        print(f"[WARNING] No valid contour found: {json_file}")
        return

    # Full Slab outline + directional boundary lines
    slab_contour, north_points, south_points, west_points, east_points = get_directional_lines(contour)

    # Visualize the Slab and directional lines with Matplotlib, saved to a file
    base_filename = os.path.splitext(os.path.basename(json_file))[0]
    viz_dir = "Drawings/FP/viz4_FP"
    os.makedirs(viz_dir, exist_ok=True)
    save_path = os.path.join(viz_dir, f"{base_filename}.png")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_facecolor('white')

    points = contour.squeeze()
    ax.plot(points[:, 0], points[:, 1], color="black", linewidth=1.5, label="Outermost contour")

    north_array = np.array([[x, y] for x, y in north_points.items()])
    south_array = np.array([[x, y] for x, y in south_points.items()])
    west_array = np.array([[x, y] for y, x in west_points.items()])
    east_array = np.array([[x, y] for y, x in east_points.items()])

    if north_array.size > 0:
        ax.scatter(north_array[:, 0], north_array[:, 1], color="blue", s=10, label="North")
    if south_array.size > 0:
        ax.scatter(south_array[:, 0], south_array[:, 1], color="red", s=10, label="South")
    if east_array.size > 0:
        ax.scatter(east_array[:, 0], east_array[:, 1], color="yellow", s=10, label="East")
    if west_array.size > 0:
        ax.scatter(west_array[:, 0], west_array[:, 1], color="green", s=10, label="West")

    plt.gca().invert_yaxis()
    ax.legend()
    plt.tight_layout()

    # Saved to a file instead of shown on screen
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[SAVED] {save_path}")

    save_coordinates_to_json(json_file, slab_contour, north_points, south_points, east_points, west_points)


# Process every JSON file in the folder
input_folder = "Drawings/FP/result3_FP"

for file in os.listdir(input_folder):
    if file.endswith(".json"):
        json_path = os.path.join(input_folder, file)
        print(f"\n[PROCESSING] {json_path}")
        process_directional_lines(json_path)

#### 1.4. Outermost objects detection

In [ ]:
import os
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt

def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def extract_contours(json_data, image_size, min_area=3000):
    mask = np.zeros((image_size[1], image_size[0]), dtype=np.uint8)

    for ann in json_data.get("annotations", []):
        seg_data = ann.get("segmentation", [])
        if not seg_data:
            continue
        segments = [seg_data] if not isinstance(seg_data[0], list) else seg_data
        for seg in segments:
            try:
                points = np.array(seg).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask, [points], 255)
            except Exception:
                continue

    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filtered_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_area]
    return max(filtered_contours, key=cv2.contourArea) if filtered_contours else None

def mark_outermost_windows_doors(json_data, slab_contour, threshold=10):
    outermost_objects = []
    slab_contour_pts = slab_contour.squeeze()
    if len(slab_contour_pts.shape) != 2 or slab_contour_pts.shape[1] != 2:
        slab_contour_pts = np.array(slab_contour).reshape(-1, 2)

    for ann in json_data.get("annotations", []):
        if ann.get("category_name") not in ["Window", "Door"]:
            continue
        is_outermost = False
        min_dist = float("inf")
        segments = ann.get("segmentation", [])
        if segments and isinstance(segments[0], (int, float)):
            segments = [segments]
        for seg in segments:
            try:
                pts = np.array(seg).reshape(-1, 2).astype(np.int32)
            except Exception:
                continue
            for pt in pts:
                d = cv2.pointPolygonTest(slab_contour_pts, (int(pt[0]), int(pt[1])), True)
                if d is not None and abs(d) < min_dist:
                    min_dist = abs(d)
                if d is not None and abs(d) < threshold:
                    is_outermost = True
                    break
            if is_outermost:
                break
        ann["is_outermost"] = is_outermost
        print(f"ID {ann.get('id')} ({ann.get('category_name')}) - min dist: {min_dist:.2f}, outermost: {ann['is_outermost']}")
        if is_outermost:
            outermost_objects.append(ann)
    return outermost_objects

def visualize_annotations_to_file(json_data, slab_contour, outermost_objects, save_path, dpi=300):
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_facecolor('white')

    if slab_contour is not None:
        slab_points = slab_contour.squeeze()
        ax.plot(slab_points[:, 0], slab_points[:, 1], color="green", linewidth=2, label="Outermost contour")

    # Regular windows/doors (blue)
    for ann in json_data.get("annotations", []):
        if ann.get("category_name") in ["Window", "Door"] and ann not in outermost_objects:
            segments = ann.get("segmentation", [])
            if segments and isinstance(segments[0], (int, float)):
                segments = [segments]
            for seg in segments:
                try:
                    pts = np.array(seg).reshape(-1, 2)
                    polygon = plt.Polygon(pts, closed=True, fill=False, edgecolor='blue', linewidth=1.5)
                    ax.add_patch(polygon)
                    centroid = np.mean(pts, axis=0)
                    ax.text(centroid[0], centroid[1], str(ann.get("id", "")), color='blue', fontsize=8, weight='bold')
                except Exception:
                    continue

    # Outermost windows/doors (red)
    for ann in outermost_objects:
        segments = ann.get("segmentation", [])
        if segments and isinstance(segments[0], (int, float)):
            segments = [segments]
        for seg in segments:
            try:
                pts = np.array(seg).reshape(-1, 2)
                polygon = plt.Polygon(pts, closed=True, fill=False, edgecolor='red', linewidth=3)
                ax.add_patch(polygon)
                centroid = np.mean(pts, axis=0)
                ax.text(centroid[0], centroid[1], str(ann.get("id", "")), color='red', fontsize=10, weight='bold')
            except Exception:
                continue

    ax.set_title("Contour (Green) & Windows/Doors (Blue) & Outermost Windows/Doors (Red)")
    ax.set_xlabel("X Coordinate")
    ax.set_ylabel("Y Coordinate")
    plt.gca().invert_yaxis()
    ax.legend(["Outermost contour", "Windows/Doors", "Outermost Windows/Doors"])

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, bbox_inches="tight", pad_inches=0, dpi=dpi)
    plt.close(fig)
    print(f"[IMG] saved -> {save_path}")

# ----------------- Single-file run -----------------
def run_once(json_file, out_json_dir, out_img_dir, threshold=30, min_area=3000, dpi=300):
    json_data = load_json(json_file)
    image_size = (
        json_data["images"][0]["width"],
        json_data["images"][0]["height"]
    )

    slab_contour = extract_contours(json_data, image_size, min_area=min_area)
    if slab_contour is None:
        print(f"[ERROR] No slab contour found: {json_file}")
        return False

    outermost = mark_outermost_windows_doors(json_data, slab_contour, threshold=threshold)

    base = os.path.splitext(os.path.basename(json_file))[0]
    out_json = os.path.join(out_json_dir, f"{base}_outermost.json")
    out_img  = os.path.join(out_img_dir,  f"{base}_outermost.png")
    os.makedirs(out_json_dir, exist_ok=True)
    os.makedirs(out_img_dir, exist_ok=True)

    # Save the outermost objects to their own JSON file
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(outermost, f, indent=4, ensure_ascii=False)
    print(f"[SAVED] Outermost-object JSON: {out_json}")

    visualize_annotations_to_file(json_data, slab_contour, outermost, out_img, dpi=dpi)
    return True

# ----------------- Batch-process a folder -----------------
def process_folder(input_dir, out_json_dir, out_img_dir,
                   threshold=30, min_area=3000, dpi=300,
                   recursive=True, keep_structure=True):
    """
    Process every .json file in input_dir:
      - save the outermost window/door list to *_outermost.json
      - save the visualization to *_outermost.png
    """
    tasks = []
    if recursive:
        for root, _, files in os.walk(input_dir):
            for fn in files:
                if fn.lower().endswith(".json"):
                    src = os.path.join(root, fn)
                    if keep_structure:
                        rel = os.path.relpath(src, input_dir)
                        stem = os.path.splitext(rel)[0]
                        dst_json = os.path.join(out_json_dir, stem + "_outermost.json")
                        dst_img  = os.path.join(out_img_dir,  stem + "_outermost.png")
                    else:
                        base = os.path.splitext(fn)[0]
                        dst_json = os.path.join(out_json_dir, base + "_outermost.json")
                        dst_img  = os.path.join(out_img_dir,  base + "_outermost.png")
                    tasks.append((src, dst_json, dst_img))
    else:
        for fn in os.listdir(input_dir):
            if fn.lower().endswith(".json"):
                src = os.path.join(input_dir, fn)
                base = os.path.splitext(fn)[0]
                dst_json = os.path.join(out_json_dir, base + "_outermost.json")
                dst_img  = os.path.join(out_img_dir,  base + "_outermost.png")
                tasks.append((src, dst_json, dst_img))

    if not tasks:
        print("[INFO] No JSON files to process.")
        return

    print(f"[INFO] Processing {len(tasks)} files...")
    ok, fail = 0, 0
    for i, (src, dst_json, dst_img) in enumerate(tasks, 1):
        try:
            print(f"[{i}/{len(tasks)}] {src}")
            os.makedirs(os.path.dirname(dst_json), exist_ok=True)
            os.makedirs(os.path.dirname(dst_img), exist_ok=True)
            # Reuses run_once's logic but with output paths fixed ahead of time
            res = _run_once_with_fixed_outputs(src, dst_json, dst_img,
                                               threshold=threshold, min_area=min_area, dpi=dpi)
            ok += 1 if res else 0
            fail += 0 if res else 1
        except Exception as e:
            print(f"[ERROR] {src}: {e}")
            fail += 1
    print(f"[DONE] success={ok}, fail={fail}")

def _run_once_with_fixed_outputs(json_file, out_json, out_img, threshold=30, min_area=3000, dpi=300):
    json_data = load_json(json_file)
    image_size = (
        json_data["images"][0]["width"],
        json_data["images"][0]["height"]
    )
    slab_contour = extract_contours(json_data, image_size, min_area=min_area)
    if slab_contour is None:
        print(f"[ERROR] No slab contour found: {json_file}")
        return False
    outermost = mark_outermost_windows_doors(json_data, slab_contour, threshold=threshold)

    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(outermost, f, indent=4, ensure_ascii=False)
    print(f"[SAVED] File saved: {out_json}")

    visualize_annotations_to_file(json_data, slab_contour, outermost, out_img, dpi=dpi)
    return True


# Batch-process the folder
process_folder(
    input_dir="Drawings/FP/result3_FP",
    out_json_dir="Drawings/FP/result5_FP",
    out_img_dir="Drawings/viz5_FP",
    threshold=30,
    min_area=3000,
    dpi=600,
    recursive=True,
    keep_structure=True
)

#### 1.5. Directional outermost objects identification

In [ ]:
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

def load_json(file_path):
    """Load a JSON file."""
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

def extract_contours(json_data, image_size, min_area=3000):
    """Extract contours from JSON data and drop small ones."""
    mask = np.zeros((image_size[1], image_size[0]), dtype=np.uint8)

    for ann in json_data["annotations"]:
        seg_data = ann.get("segmentation", [])
        if not seg_data:
            continue
        if not isinstance(seg_data[0], list):
            segments = [seg_data]
        else:
            segments = seg_data
        for seg in segments:
            try:
                points = np.array(seg).reshape(-1, 2)
                points = points.astype(np.int32)
                cv2.fillPoly(mask, [points], 255)
            except Exception:
                continue

    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    filtered_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_area]
    if filtered_contours:
        return max(filtered_contours, key=cv2.contourArea)
    return None

def get_directional_lines(contour):
    """Compute the directional boundary lines and the full Slab outline (stricter version)."""
    points = contour.squeeze()

    north_points = {}
    south_points = {}
    east_points = {}
    west_points = {}
    slab_contour = []

    x_coords = points[:, 0]
    y_coords = points[:, 1]
    x_min, x_max = np.min(x_coords), np.max(x_coords)
    y_min, y_max = np.min(y_coords), np.max(y_coords)

    offset = 20  # margin from each edge

    for x, y in points:
        slab_contour.extend([int(x), int(y)])
        if y <= y_min + offset:
            north_points[(x, y)] = True
        if y >= y_max - offset:
            south_points[(x, y)] = True
        if x >= x_max - offset:
            east_points[(x, y)] = True
        if x <= x_min + offset:
            west_points[(x, y)] = True

    print(f"North points count: {len(north_points)}")
    print(f"South points count: {len(south_points)}")
    print(f"East points count: {len(east_points)}")
    print(f"West points count: {len(west_points)}")

    return slab_contour, north_points, south_points, east_points, west_points

def save_coordinates_to_json(json_file, slab_contour, north_points, south_points, east_points, west_points):
    """Save the directional lines and full Slab outline to JSON, correctly ordered."""
    base_filename = os.path.splitext(os.path.basename(json_file))[0]
    output_path = f"directional_coordinates_{base_filename}.json"

    directional_data = {
        "Slab": [int(coord) for coord in slab_contour],
        "North": [int(coord) for (x, y) in sorted(north_points.keys()) for coord in (x, y)],
        "South": [int(coord) for (x, y) in sorted(south_points.keys()) for coord in (x, y)],
        "East": [int(coord) for (x, y) in sorted(east_points.keys(), key=lambda p: p[1]) for coord in (x, y)],
        "West": [int(coord) for (x, y) in sorted(west_points.keys(), key=lambda p: p[1]) for coord in (x, y)]
    }

    with open(output_path, "w") as f:
        json.dump(directional_data, f, indent=4)

    print(f"[SAVED] {output_path}")

# ------------------------------------------------------------------
# Core logic: assign each outermost object a direction by majority vote
# ------------------------------------------------------------------

def find_direction_by_majority(segmentation, directional_data):
    directions = ["North", "South", "East", "West"]

    dir_points = {}
    for d in directions:
        arr = directional_data.get(d, [])
        arr_np = np.array(arr).reshape(-1, 2) if len(arr) > 0 else np.empty((0, 2))
        dir_points[d] = arr_np

    if all(dir_points[d].size == 0 for d in directions):
        print("[WARNING] No directional points found at all!")
        return None

    if segmentation and isinstance(segmentation[0], (int, float)):
        segmentation = [segmentation]

    direction_counts = {d: 0 for d in directions}
    total_points = 0

    for seg in segmentation:
        try:
            pts = np.array(seg).reshape(-1, 2)
        except Exception:
            continue
        for pt in pts:
            best_dir = None
            min_dist = float('inf')
            for d in directions:
                if dir_points[d].size == 0:
                    continue
                distances = np.linalg.norm(dir_points[d] - pt, axis=1)
                d_min = np.min(distances) if len(distances) else float('inf')

                if d_min < min_dist:
                    min_dist = d_min
                    best_dir = d

            if best_dir is not None:
                direction_counts[best_dir] += 1
            total_points += 1

    if total_points == 0:
        return None

    max_count = max(direction_counts.values())
    winners = [d for d, cnt in direction_counts.items() if cnt == max_count]

    if len(winners) == 1:
        return winners[0]
    else:
        return winners[0]

def assign_direction_to_outermost(outermost_objects, directional_data):
    """Assign each outermost object a direction (N/S/E/W) by majority vote."""
    for obj in outermost_objects:
        seg = obj.get("segmentation", [])
        direction = find_direction_by_majority(seg, directional_data)
        obj["direction"] = direction if direction else "Unknown"
    return outermost_objects

# ------------------------------------------------------------------

def visualize_final_results(outermost_objects, directional_data, slab_contour, save_path=None):
    """Visualize the Slab contour, directional lines, and outermost objects (saved, not shown)."""
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_facecolor('white')

    slab_points = np.array(slab_contour).reshape(-1, 2)
    ax.plot(slab_points[:, 0], slab_points[:, 1], linewidth=2, color="black", label="Outermost contour")

    north_array = np.array(directional_data["North"]).reshape(-1, 2)
    south_array = np.array(directional_data["South"]).reshape(-1, 2)
    east_array = np.array(directional_data["East"]).reshape(-1, 2)
    west_array = np.array(directional_data["West"]).reshape(-1, 2)

    if north_array.size > 0:
        ax.scatter(north_array[:, 0], north_array[:, 1], s=10, label="North")
    if south_array.size > 0:
        ax.scatter(south_array[:, 0], south_array[:, 1], s=10, label="South")
    if east_array.size > 0:
        ax.scatter(east_array[:, 0], east_array[:, 1], s=10, label="East")
    if west_array.size > 0:
        ax.scatter(west_array[:, 0], west_array[:, 1], s=10, label="West")

    for obj in outermost_objects:
        direction = obj.get("direction", "Unknown")
        seg = obj.get("segmentation", [])
        if seg and isinstance(seg[0], (int, float)):
            seg = [seg]
        for seg_part in seg:
            try:
                pts = np.array(seg_part).reshape(-1, 2)
            except Exception:
                continue
            polygon = plt.Polygon(pts, closed=True, fill=False, edgecolor='red', linewidth=2)
            ax.add_patch(polygon)
            centroid = np.mean(pts, axis=0)
            ax.text(centroid[0], centroid[1], direction, color='red', fontsize=10, weight='bold')

    ax.set_title("Contour + Directional Lines + Outermost Windows/Doors")
    ax.set_xlabel("X Coordinate")
    ax.set_ylabel("Y Coordinate")
    plt.gca().invert_yaxis()
    ax.legend(["Outermost contour", "North", "South", "East", "West", "Outermost Objects"])

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight", dpi=300)
        plt.close(fig)
        print(f"[SAVED] image: {save_path}")
    else:
        # Kept for manual/interactive testing only
        plt.show()
        plt.close(fig)

def assign_and_visualize(directional_json_path, outermost_json_path, final_output_path, viz_output_path=None):
    """Assign directions and render the visualization for one drawing."""
    directional_data = load_json(directional_json_path)
    slab_contour = directional_data["Slab"]

    outermost_objects = load_json(outermost_json_path)
    updated_outermost_objects = assign_direction_to_outermost(outermost_objects, directional_data)

    with open(final_output_path, 'w') as f:
        json.dump(updated_outermost_objects, f, indent=4)
    print(f"[SAVED] JSON file: {final_output_path}")

    visualize_final_results(updated_outermost_objects, directional_data, slab_contour, save_path=viz_output_path)

# =========================
# Batch loop over the folder (normalizes filenames, adds the viz output path)
# =========================
dir_direction = "Drawings/FP/result4_FP"   # folder of directional-coordinate JSONs (e.g. TEST_1_direction.json)
dir_outermost = "Drawings/FP/result5_FP"   # folder of *_outermost.json files (e.g. TEST_1_outermost.json)
dir_output    = "Drawings/FP/result6_FP"   # folder for the final JSON output
dir_viz       = "Drawings/FP/viz6_FP"      # folder for the visualization images

os.makedirs(dir_output, exist_ok=True)
os.makedirs(dir_viz, exist_ok=True)

files = [f for f in os.listdir(dir_direction) if f.lower().endswith(".json")]
if not files:
    print(f"[WARNING] No files to process: {dir_direction}")

for fn in files:
    base = os.path.splitext(fn)[0]  # e.g. TEST_1_direction
    base_core = base

    # Normalize the filename by stripping known suffixes/prefixes
    for suf in ["_direction", "_directions", "_dir", "_directional"]:
        if base_core.endswith(suf):
            base_core = base_core[: -len(suf)]
    if base_core.startswith("directional_coordinates_"):
        base_core = base_core[len("directional_coordinates_"):]

    directional_json_path = os.path.join(dir_direction, fn)
    outermost_json_path   = os.path.join(dir_outermost, f"{base_core}_outermost.json")
    final_output_path     = os.path.join(dir_output,   f"{base_core}.json")
    viz_output_path       = os.path.join(dir_viz,      f"{base_core}.png")

    print(f"\n[MAPPING]\n  direction: {directional_json_path}\n  outermost: {outermost_json_path}\n  output   : {final_output_path}\n  viz      : {viz_output_path}")

    if not os.path.exists(outermost_json_path):
        print(f"[SKIP] No matching outermost JSON, skipping: {outermost_json_path}")
        continue

    print(f"[PROCESSING] {base_core}")
    assign_and_visualize(directional_json_path, outermost_json_path, final_output_path, viz_output_path)

#### 1.6. Update object recognition result (Final json file)

In [ ]:
import json
import os

def load_json(file_path):
    """Load a JSON file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def update_existing_json(existing_json_path, final_json_path, output_json_path):
    """Update an existing JSON file with the final direction/is_outermost results."""
    existing_data = load_json(existing_json_path)
    final_data = load_json(final_json_path)

    # The final file may be a list, or a dict with an "annotations" key
    if "annotations" in final_data:
        final_annotations = final_data["annotations"]
    else:
        final_annotations = final_data

    final_annotations_dict = {ann["id"]: ann for ann in final_annotations}

    for ann in existing_data["annotations"]:
        obj_id = ann["id"]
        if obj_id in final_annotations_dict:
            ann["is_outermost"] = True
            ann["direction"] = final_annotations_dict[obj_id].get("direction", "Unknown")
        else:
            ann["is_outermost"] = False
            ann["direction"] = "Unknown"

    os.makedirs(os.path.dirname(output_json_path), exist_ok=True)
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(existing_data, f, indent=4, ensure_ascii=False)
    print(f"[SAVED] Output JSON: {output_json_path}")

# =========================
# Batch loop over the folder (logic unchanged)
# =========================
dir_existing = "Drawings/FP/result3_FP"   # folder of pre-existing JSONs
dir_final    = "Drawings/FP/result6_FP"   # folder of final JSONs with is_outermost & direction
dir_output   = "Drawings/FP/result7_FP"   # folder for the updated JSON output

os.makedirs(dir_output, exist_ok=True)

files = [f for f in os.listdir(dir_existing) if f.lower().endswith(".json")]
if not files:
    print(f"[WARNING] No files to process: {dir_existing}")

for fn in files:
    base = os.path.splitext(fn)[0]
    existing_json_path = os.path.join(dir_existing, fn)
    final_json_path    = os.path.join(dir_final, fn)     # matched by identical filename
    output_json_path   = os.path.join(dir_output, fn)    # saved under the same filename

    if not os.path.exists(final_json_path):
        print(f"[SKIP] No matching final JSON, skipping: {final_json_path}")
        continue

    print(f"\n[MERGED] {base}.json")
    update_existing_json(existing_json_path, final_json_path, output_json_path)

# 2. Elevation recognition

Detects Wall/Window/Door objects in elevation drawings, clusters them into floors along the vertical axis, and converts image coordinates into the building's real-world coordinate system.


#### 2.1. YOLO11s-seg + SAHI-based elevation object recognition

In [ ]:
import os
import json
import cv2
import numpy as np
from collections import defaultdict
from sahi.predict import get_sliced_prediction
from ultralytics import YOLO
from sahi.models.ultralytics import UltralyticsDetectionModel
from matplotlib.path import Path

def process_images_in_directory(image_dir, model_path, output_json_dir):
    """YOLO + SAHI based elevation object recognition, saved as JSON (segmentation included)."""
    model = YOLO(model_path)
    class_names = model.names

    # Wrap the same YOLO model for SAHI sliced inference
    detection_model = UltralyticsDetectionModel(
        model_path=model_path,
        confidence_threshold=0.3,
        device="cuda"
    )

    os.makedirs(output_json_dir, exist_ok=True)

    skip_log = []

    for image_file in os.listdir(image_dir):
        if not image_file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        image_path = os.path.join(image_dir, image_file)
        direction = extract_direction_from_filename(image_file)
        if not direction:
            print(f"Skipping file (no direction found): {image_file}")
            continue

        print(f"Processing image: {image_path} (Direction: {direction})")

        img = cv2.imread(image_path)
        if img is None:
            print(f"Failed to read image: {image_path}")
            continue
        H_original, W_original = img.shape[:2]

        # Run YOLO prediction (detects the Wall class)
        results = model.predict(source=img.copy(), save=False, save_txt=False, stream=True, batch=32)

        # Process Wall-class detections
        coco_annotations = []
        image_id = os.path.splitext(image_file)[0]
        wall_category_id = [key for key, value in class_names.items() if value == "Wall"][0]

        for result in results:
            if result.masks is None or result.boxes is None:
                continue

            masks = result.masks.data.cpu().numpy().astype('uint8')
            scores = result.boxes.conf.cpu().numpy()
            class_ids = result.boxes.cls.cpu().numpy().astype(int)

            N, H_scaled, W_scaled = masks.shape
            scale_x = W_original / W_scaled
            scale_y = H_original / H_scaled

            for i in range(N):
                if class_names[class_ids[i]] != "Wall":
                    continue

                mask = masks[i]
                score = scores[i]
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                polygon = []
                total_area = 0
                for contour in contours:
                    coords_scaled = contour[:, 0, :]
                    coords_original = coords_scaled * [scale_x, scale_y]
                    coords_original = coords_original.astype(int).tolist()
                    polygon.append(coords_original)
                    total_area += cv2.contourArea(coords_scaled)

                if polygon:
                    flat_polygon = [coord for poly in polygon for point in poly for coord in point]
                    annotation = {
                        "image_id": image_id,
                        "bbox": [
                            float(min([p[0] for poly in polygon for p in poly])),
                            float(min([p[1] for poly in polygon for p in poly])),
                            float(max([p[0] for poly in polygon for p in poly]) - min([p[0] for poly in polygon for p in poly])),
                            float(max([p[1] for poly in polygon for p in poly]) - min([p[1] for poly in polygon for p in poly]))
                        ],
                        "score": float(score),
                        "category_id": int(wall_category_id),
                        "category_name": "Wall",
                        "segmentation": [flat_polygon],
                        "iscrowd": 0,
                        "area": float(total_area)
                    }
                    coco_annotations.append(annotation)

        # SAHI-based Window and Door detection
        try:
            sahi_results = get_sliced_prediction(
                image=image_path,
                detection_model=detection_model,
                slice_height=1024,
                slice_width=1024,
                overlap_height_ratio=0.2,
                overlap_width_ratio=0.2
            )
            sahi_annotations = sahi_results.to_coco_annotations()

            for obj in sahi_annotations:
                if obj["category_name"] in ["Window", "Door"]:
                    bbox_x_min, bbox_y_min, bbox_width, bbox_height = obj["bbox"]

                    coco_annotations.append({
                        "image_id": image_id,
                        "bbox": obj["bbox"],
                        "width": bbox_width,   # Window/Door only
                        "height": bbox_height, # Window/Door only
                        "score": obj["score"],
                        "category_id": obj["category_id"],
                        "category_name": obj["category_name"],
                        "segmentation": obj["segmentation"],
                        "iscrowd": obj.get("iscrowd", 0),
                        "area": obj.get("area", 0)
                    })
        except ValueError as e:
            print(f"Skipping invalid mask in image {image_file}: {str(e)}")
            skip_log.append({
                "image_file": image_file,
                "error": str(e)
            })

        # Transform center coordinates into the building's coordinate system
        building_width = calculate_building_width(coco_annotations)
        processed_annotations = process_annotations(coco_annotations, direction, building_width)

        output_data = {
            "images": [{"id": image_id, "file_name": image_file, "height": H_original, "width": W_original}],
            "annotations": processed_annotations
        }
        output_json_path = os.path.join(output_json_dir, f"{image_id}.json")
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=4, ensure_ascii=False)
        print(f"Saved transformed coordinates to {output_json_path}")

    if skip_log:
        log_path = os.path.join(output_json_dir, "skip_log.json")
        with open(log_path, "w", encoding="utf-8") as f:
            json.dump(skip_log, f, indent=4, ensure_ascii=False)
        print(f"Saved skip log to {log_path}")

def extract_direction_from_filename(filename):
    """Extract the direction (South, East, North, West) from a filename."""
    directions = ["South", "East", "North", "West"]
    for direction in directions:
        if direction.lower() in filename.lower():
            return direction
    return None

def calculate_building_width(object_data):
    """Estimate the building width from the X-coordinate range of detected objects."""
    all_x = []
    for obj in object_data:
        bbox = obj["bbox"]
        x_min, width = bbox[0], bbox[2]
        x_max = x_min + width
        all_x.extend([x_min, x_max])

    building_width = max(all_x) - min(all_x)
    return building_width

def transform_center(direction, x_center, y_center, building_width):
    """Transform a center coordinate according to elevation direction."""
    if direction == "South":
        return [x_center, None, y_center]
    elif direction == "East":
        return [None, x_center, y_center]
    elif direction == "North":
        return [building_width - x_center, None, y_center]
    elif direction == "West":
        return [None, building_width - x_center, y_center]
    else:
        raise ValueError("Invalid direction.")

def process_annotations(object_data, direction, building_width):
    """Transform center coordinates per direction and record width/height."""
    processed_data = []
    for obj_id, obj in enumerate(object_data, start=1):
        x_min, y_min, width, height = obj["bbox"]
        x_center = x_min + width / 2
        y_center = y_min + height / 2

        transformed_center = transform_center(direction, x_center, y_center, building_width)

        annotation = {**obj, "id": obj_id, "original_center": [x_center, y_center], "transformed_center": transformed_center}
        processed_data.append(annotation)

    return processed_data

image_dir = "Drawings/ED/Images"

# Pre-trained model weights are not included in this repository.
# Please specify the path to your own compatible model weights.
model_path = "path/to/elevation_model.pt"

output_json_dir = "Drawings/ED/result_ED"

process_images_in_directory(image_dir, model_path, output_json_dir)

##### Visualization

Renders the detected elevation objects on top of the source image and saves the result to disk.


In [ ]:
import os
import json
import cv2
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Polygon
import random

def visualize_segmentation(image_path, json_path, output_path=None, dark_mode=False, use_bbox=False):
    """Visualize a single image + JSON annotation file and save the result."""
    img = cv2.imread(image_path)
    if img is None:
        print(f"[WARNING] Image not found: {image_path}")
        return

    if dark_mode:
        img = np.zeros_like(img)  # black background

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # OpenCV is BGR, Matplotlib expects RGB

    if not os.path.exists(json_path):
        print(f"[WARNING] JSON not found: {json_path}")
        return
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    annotations = data.get("annotations", [])

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img)

    def random_color():
        return [random.random(), random.random(), random.random()]

    for ann in annotations:
        category_id = ann.get("category_id")
        if category_id not in [1]:  # Window and Door only (extend to [1, 2] if needed)
            continue

        if use_bbox:
            bbox = ann.get("bbox", [])
            if bbox:
                x_min, y_min, width, height = bbox
                x_max = x_min + width
                y_max = y_min + height
                coords = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])
                color = random_color()
                polygon = Polygon(coords, closed=True, edgecolor=color, facecolor='none', linewidth=0.8)
                ax.add_patch(polygon)
        else:
            segmentation = ann.get("segmentation", [])
            if not segmentation:
                continue
            for segment in segmentation:
                coords = np.array(segment).reshape(-1, 2)
                color = random_color()
                polygon = Polygon(coords, closed=True, edgecolor=color, facecolor='none', linewidth=0.8)
                ax.add_patch(polygon)

    ax.axis("off")

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, bbox_inches="tight", pad_inches=0, dpi=300)
        plt.close(fig)
        print(f"[SAVED] visualization -> {output_path}")
    else:
        plt.show()

# ===============================
# Batch loop over the folder
# ===============================
dir_images = "Drawings/ED/Images"     # source images
dir_jsons  = "Drawings/ED/result_ED"  # JSON results
dir_output = "Drawings/ED/viz_ED"     # visualization output
os.makedirs(dir_output, exist_ok=True)

image_files = [f for f in os.listdir(dir_images) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

if not image_files:
    print(f"[WARNING] No image files found in: {dir_images}")

for img_file in image_files:
    base_name = os.path.splitext(img_file)[0]
    image_path = os.path.join(dir_images, img_file)
    json_path  = os.path.join(dir_jsons, f"{base_name}.json")
    output_path = os.path.join(dir_output, f"{base_name}.png")

    print(f"\n[PROCESSING] {base_name}")
    visualize_segmentation(image_path, json_path, output_path, dark_mode=False, use_bbox=False)

#### 2.2. Floor-wise clustering

In [ ]:
import os
import json
import numpy as np
from collections import defaultdict
from sklearn.cluster import DBSCAN

##############################
# 1) Load JSON data
##############################

def load_json_data(json_dir):
    """
    Load every .json file under json_dir:
      - keep every object in the "annotations" list
      - for Window/Door objects (category_id in [0, 1]) only, also filter
        on transformed_center[2] (the z value)

    Returns:
      data: { filename.json: {"z_values": ..., "annotations": ...} }  # Window & Door only
      metadata: { filename.json: {"images": [...], "annotations": [...]} }  # everything kept
    """
    data = {}
    metadata = {}
    for json_file in os.listdir(json_dir):
        if not json_file.endswith(".json"):
            continue

        file_path = os.path.join(json_dir, json_file)
        with open(file_path, "r", encoding="utf-8") as f:
            file_data = json.load(f)

        # Handle both dict-shaped and list-shaped JSON
        if isinstance(file_data, dict):
            images = file_data.get("images", [])
            annotations = file_data.get("annotations", [])
        elif isinstance(file_data, list):
            images = []  # no image metadata available for a bare list
            annotations = file_data
        else:
            continue  # unrecognized JSON structure

        # Extract Window & Door objects only (used for floor assignment)
        filtered = [
            ann for ann in annotations
            if ann.get("category_id") in [0, 1]  # Window or Door
               and ann.get("transformed_center")
               and len(ann["transformed_center"]) >= 3
               and ann["transformed_center"][2] is not None
        ]

        # Keep every object, including Wall
        metadata[json_file] = {"images": images, "annotations": annotations}

        if filtered:
            z_arr = np.array([ann["transformed_center"][2] for ann in filtered]).reshape(-1, 1)
            data[json_file] = {"z_values": z_arr, "annotations": filtered}

    return data, metadata

##############################
# 2) DBSCAN clustering + floor assignment
##############################

def assign_floors_with_dbscan(z_arr, annotations, eps=50, min_samples=5):
    """
    Assign a floor number to each Window/Door object using DBSCAN clustering.
    1) Drop noise points (label -1)
    2) Rank clusters by mean z value (descending) and number them from floor 1
    3) annotation["floor"] = floor_num

    Returns:
      valid_annotations: the Window/Door objects that received a floor number
    """
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(z_arr)

    # Keep only objects that belong to a cluster
    valid_annotations = [ann for ann, label in zip(annotations, labels) if label != -1]

    if not valid_annotations:
        print("[WARNING] No valid clusters found. Skipping floor assignment.")
        return []

    # Compute the mean z value per cluster
    cluster_map = defaultdict(list)
    for idx, label in enumerate(labels):
        if label == -1:  # ignore noise
            continue
        cluster_map[label].append(idx)

    cluster_mean = {lbl: np.mean([annotations[i]["transformed_center"][2] for i in indices]) for lbl, indices in cluster_map.items()}

    # Higher z value -> lower floor number
    sorted_labels = sorted(cluster_mean, key=lambda l: cluster_mean[l], reverse=True)
    label_to_floor = {lbl: (i + 1) for i, lbl in enumerate(sorted_labels)}

    # Only Window/Door objects get a floor value
    for idx, label in enumerate(labels):
        if label == -1:
            continue
        annotations[idx]["floor"] = label_to_floor[label]

    return valid_annotations

##############################
# 3) Save JSON (Wall kept as-is, unclustered Window/Door removed)
##############################

def save_json_with_floors(file_name, images, original_annotations, updated_annotations, output_dir):
    """
    Save the JSON while preserving its original structure:
    - drop Window/Door objects that did not belong to a cluster
    - keep every Wall object
    - add floor info to the remaining Window/Door objects
    """
    out_path = os.path.join(output_dir, file_name)

    # Keep Wall objects as-is; Window/Door objects reflect the clustering result
    updated_ann_dict = {ann["id"]: ann for ann in updated_annotations}
    new_annotations = [
        obj for obj in original_annotations
        if obj.get("category_id") == 2 or obj.get("id") in updated_ann_dict  # Wall kept, Window/Door filtered
    ]

    output_data = {
        "images": images,
        "annotations": new_annotations
    }

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"[SAVED] Floor-added JSON saved to {out_path}")

##############################
# 4) Main entry point
##############################

def add_floors_to_json(json_dir, output_dir, eps=50, min_samples=5):
    """
    1) load_json_data
    2) apply DBSCAN clustering (Window/Door only)
    3) save JSON (original structure kept, floor added, unclustered Window/Door removed)
    """
    os.makedirs(output_dir, exist_ok=True)

    data, metadata = load_json_data(json_dir)

    for file_name, content in data.items():
        print(f"\n=== Processing {file_name} ===")

        z_arr = content["z_values"]
        window_door_anns = content["annotations"]

        images = metadata[file_name]["images"]
        original_annotations = metadata[file_name]["annotations"]

        updated_anns = assign_floors_with_dbscan(z_arr, window_door_anns, eps, min_samples)

        if updated_anns:
            save_json_with_floors(file_name, images, original_annotations, updated_anns, output_dir)
        else:
            print(f"[ERROR] {file_name}: No valid floor assignments, skipping file.")

    print(f"\n[DONE] All files saved: ({output_dir})")

##############################
# Run
##############################

json_dir = "Drawings/ED/result_ED"
output_dir = "Drawings/ED/result2_ED"

add_floors_to_json(json_dir, output_dir, eps=50, min_samples=3)

##### Visualization

Renders each object's floor assignment (color-coded) on top of the source image and saves the result to disk.


In [ ]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def visualize_original_centers_by_floor(json_path, image_path, output_path=None):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"[ERROR] JSON not found: {json_path}")
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict) and "annotations" in data:
        annotations = data["annotations"]
    elif isinstance(data, dict):  # dict keyed by floor number
        annotations = []
        for v in data.values():
            annotations.extend(v)
    elif isinstance(data, list):
        annotations = data
    else:
        raise ValueError("[ERROR] Invalid JSON structure")

    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"[ERROR] Cannot read image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(img)

    # Assign a color to each distinct floor
    floor_set = sorted(set(
        ann.get("floor") for ann in annotations
        if ann.get("original_center") and ann.get("floor") is not None
    ))

    floor_colors = {floor: np.random.rand(3,) for floor in floor_set}

    for ann in annotations:
        center = ann.get("original_center")
        ann_id = ann.get("id", "N/A")
        floor = ann.get("floor")
        if not center or center[0] is None or center[1] is None or floor is None:
            continue
        x, y = center
        color = floor_colors.get(floor, (1, 0, 0))  # fallback = red

        ax.plot(x, y, 'o', color=color, markersize=3)

    legend_patches = [
        mpatches.Patch(color=color, label=f"Floor {floor}")
        for floor, color in floor_colors.items()
    ]
    ax.legend(handles=legend_patches, loc="upper right", fontsize=9)

    ax.axis("off")

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, bbox_inches='tight', dpi=300)
        plt.close()
        print(f"[SAVED] {output_path}")
    else:
        plt.show()

def batch_visualize_centers_by_floor(json_dir, image_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for json_file in os.listdir(json_dir):
        if not json_file.endswith('.json'):
            continue
        base_name = os.path.splitext(json_file)[0]
        json_path = os.path.join(json_dir, json_file)
        image_path = os.path.join(image_dir, base_name + ".png")
        output_path = os.path.join(output_dir, f"{base_name}.png")

        visualize_original_centers_by_floor(json_path, image_path, output_path)

batch_visualize_centers_by_floor(
    json_dir="Drawings/ED/result2_ED",
    image_dir="Drawings/ED/Images",
    output_dir="Drawings/ED/viz2_ED"
)

 #### 2.3. Coordinate system conversion
- The image coordinate system has its origin at the top-left with y increasing downward, while the real-world coordinate system has y increasing upward. Only the y-axis needs to be flipped when converting between them.


In [ ]:
import os
import json
from glob import glob

def adjust_image_coordinates(ed_data):
    """
    Convert an elevation drawing's y-coordinates into the building's
    real-world coordinate system.
    """

    # 1) Find the highest y value among Wall objects
    wall_y_coords = []

    for obj in ed_data["annotations"]:
        if obj.get("category_name") == "Wall":  # Wall objects only
            bbox = obj.get("bbox", [0, 0, 0, 0])
            y_min = bbox[1]
            wall_y_coords.append(y_min)

            original_center = obj.get("original_center", [0, 0])
            wall_y_coords.append(original_center[1])

            transformed_center = obj.get("transformed_center", [0, 0, 0])
            wall_y_coords.append(transformed_center[2])  # z value of transformed_center

            segmentation = obj.get("segmentation", [])
            for seg in segmentation:
                wall_y_coords.extend(seg[1::2])  # every y coordinate

    if not wall_y_coords:
        print("[WARNING] No valid Wall objects found in ED data.")
        return ed_data

    y_max = max(wall_y_coords)

    # 2) Apply the coordinate transform to every object
    for obj in ed_data["annotations"]:
        # bbox y_min
        bbox = obj.get("bbox", [0, 0, 0, 0])
        obj["bbox"][1] = y_max - bbox[1]

        # original_center
        original_center = obj.get("original_center", [0, 0])
        obj["original_center"] = [original_center[0], y_max - original_center[1]]

        # transformed_center (z value)
        transformed_center = obj.get("transformed_center", [0, 0, 0])
        obj["transformed_center"] = [transformed_center[0], transformed_center[1], y_max - transformed_center[2]]

        # segmentation coordinates
        new_segmentation = []
        for seg in obj.get("segmentation", []):
            new_seg = []
            for i in range(0, len(seg), 2):
                x = seg[i]
                y = seg[i + 1]
                new_y = y_max - y
                new_seg.extend([x, new_y])
            new_segmentation.append(new_seg)

        obj["segmentation"] = new_segmentation

    return ed_data

def process_all_ed_files(input_folder, output_folder):
    """Apply the coordinate transform to every ED JSON file in a folder and save the results."""
    os.makedirs(output_folder, exist_ok=True)
    ed_files = glob(os.path.join(input_folder, "*.json"))

    for ed_file in ed_files:
        ed_filename = os.path.basename(ed_file)
        output_file = os.path.join(output_folder, ed_filename.replace(".json", "_final.json"))

        print(f"\n[PROCESSING] {ed_filename}")

        with open(ed_file, "r", encoding="utf-8") as f:
            ed_data = json.load(f)

        adjusted_ed_data = adjust_image_coordinates(ed_data)

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(adjusted_ed_data, f, indent=4)

        print(f"[SAVED] {output_file}")

##############################
# Run
##############################

input_folder = "Drawings/ED/result2_ED"
output_folder = "Drawings/ED/result3_ED"

process_all_ed_files(input_folder, output_folder)

print("\n[DONE] Coordinate conversion complete for all files!")

# 3. Multi-view Correspondence matching

Matches floor-plan objects to their corresponding elevation objects (same floor, same direction) using the Hungarian algorithm, so each object gets a full 3D position.


#### 3.1. Hungarian algorithm-based multi-view correspondence matching

In [ ]:
import os
import json
import ntpath
import numpy as np
from glob import glob
from copy import deepcopy
from scipy.optimize import linear_sum_assignment

##############################
# Utils
##############################

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def get_filename_no_ext(path):
    head, tail = ntpath.split(path)
    return os.path.splitext(tail)[0]

def get_direction_from_image_id(image_id):
    lower_id = image_id.lower()
    if "north" in lower_id: return "North"
    if "south" in lower_id: return "South"
    if "east"  in lower_id: return "East"
    if "west"  in lower_id: return "West"
    return "Unknown"

##############################
# Build FP / ED dict
##############################

def build_fp_dict(fp_data):
    """Group floor-plan annotations by (floor_number, direction), outermost objects only."""
    fp_dict = {}
    for ann in fp_data.get("annotations", []):
        if not ann.get("is_outermost", False):
            continue
        floor_no  = ann.get("floor_number")
        direction = ann.get("direction", "Unknown")
        if floor_no is None or direction == "Unknown":
            continue
        key = (floor_no, direction)
        fp_dict.setdefault(key, []).append(ann)
    return fp_dict

def build_ed_dict(ed_data):
    """Group elevation annotations by (floor, direction); direction comes from image_id."""
    ed_dict = {}
    for obj in ed_data.get("annotations", []):
        floor_no = obj.get("floor", 0)
        ed_dir   = get_direction_from_image_id(obj.get("image_id", ""))
        if ed_dir == "Unknown":
            continue
        key = (floor_no, ed_dir)
        ed_dict.setdefault(key, []).append(obj)
    return ed_dict

def merge_ed_dicts(ed_dict_list):
    """Merge multiple ED-file dicts, keyed by (floor, direction)."""
    merged = {}
    for d in ed_dict_list:
        for k, v in d.items():
            merged.setdefault(k, []).extend(v)
    return merged

##############################
# Matching (Hungarian)
##############################

def build_cost_matrix(fp_objs, ed_objs, direction):
    N, M = len(fp_objs), len(ed_objs)
    cost_mat = np.zeros((N, M), dtype=float)
    for i, fp in enumerate(fp_objs):
        x_fp, y_fp = fp["original_center"]
        for j, ed in enumerate(ed_objs):
            x_ed, y_ed, z_ed = ed.get("transformed_center", [0, 0, 0])
            dist = abs(x_fp - x_ed) if direction in ["North", "South"] else abs(y_fp - y_ed)
            cost_mat[i, j] = dist
    return cost_mat

def match_hungarian(fp_objs, ed_objs, direction):
    cost_mat = build_cost_matrix(fp_objs, ed_objs, direction)
    row_ind, col_ind = linear_sum_assignment(cost_mat)

    matched_dict    = {fp["id"]: None for fp in fp_objs}
    ed_matched_dict = {ed["id"]: None for ed in ed_objs}
    matched_pairs   = []

    for fp_idx, ed_idx in zip(row_ind, col_ind):
        fp_obj = fp_objs[fp_idx]
        ed_obj = ed_objs[ed_idx]
        matched_dict[fp_obj["id"]]   = ed_obj
        ed_matched_dict[ed_obj["id"]] = fp_obj
        matched_pairs.append((fp_obj["id"], ed_obj["id"], fp_obj["original_center"]))
    return matched_dict, ed_matched_dict, matched_pairs

##############################
# Update functions
##############################

def update_fp_transformed_center(fp_data, matched_dict):
    """Window/Door take the bottom z; everything else keeps z_center. Height is also recorded."""
    for obj in fp_data.get("annotations", []):
        matched_ed = matched_dict.get(obj["id"], None)
        if not matched_ed:
            continue

        z_center = matched_ed["transformed_center"][2]
        bbox     = matched_ed.get("bbox", [0, 0, 0, 0])
        height   = bbox[3] if len(bbox) >= 4 else 0

        if obj["category_name"].lower() in ["window", "door"]:
            bottom_z = z_center - (height / 2) if height > 0 else z_center
            obj["transformed_center"][2] = bottom_z
        else:
            obj["transformed_center"][2] = z_center

        obj["height"]  = height
        obj["matched"] = matched_ed["id"]
    return fp_data

##############################
# Main (SINGLE + CUMULATIVE)
##############################

def match_all_floors_and_directions():
    fp_dir      = "Drawings/FP/result7_FP"
    ed_dir      = "Drawings/ED/result3_ED"
    out_root    = "Drawings/Matched/outputs"           # root output folder
    out_single  = os.path.join(out_root, "Single")      # per-direction single result
    out_cum     = os.path.join(out_root, "ALL")         # cumulative result

    os.makedirs(out_single, exist_ok=True)
    os.makedirs(out_cum, exist_ok=True)

    fp_files = glob(os.path.join(fp_dir, "*.json"))
    ed_files = glob(os.path.join(ed_dir, "*.json"))

    # 1) Read every ED file and merge into one dict, keyed by (floor, direction)
    ed_dicts = []
    for ed_file in ed_files:
        ed_data = load_json(ed_file)
        ed_dicts.append(build_ed_dict(ed_data))
    union_ed_dict = merge_ed_dicts(ed_dicts)

    print("\n=== Matching (SINGLE + CUMULATIVE) ===")

    for fp_file in fp_files:
        fp_name   = get_filename_no_ext(fp_file)
        fp_src    = load_json(fp_file)          # original data, used for the single-direction results
        fp_accum  = deepcopy(fp_src)            # accumulates across directions

        fp_dict = build_fp_dict(fp_src)

        total_pairs = 0

        # 2) Match per (floor, direction)
        for (floor_no, direction), fp_objs in fp_dict.items():
            ed_objs = union_ed_dict.get((floor_no, direction), [])
            if not ed_objs:
                continue

            matched_dict, _, matched_pairs = match_hungarian(fp_objs, ed_objs, direction)
            pair_cnt = len(matched_pairs)
            total_pairs += pair_cnt

            # 2a) Single-direction result: updates a fresh copy of the original each time
            fp_single = deepcopy(fp_src)
            fp_single = update_fp_transformed_center(fp_single, matched_dict)
            single_path = os.path.join(out_single, f"{fp_name}_{direction}.json")
            save_json(fp_single, single_path)

            # 2b) Cumulative result: keeps updating the same copy in place
            fp_accum = update_fp_transformed_center(fp_accum, matched_dict)

            print(f"[MATCHED] {fp_name} [{direction}] matched {pair_cnt} objs -> SINGLE saved")

        # 3) Save the cumulative result once, after all directions are processed
        cum_path = os.path.join(out_cum, f"{fp_name}_ALL.json")
        save_json(fp_accum, cum_path)
        print(f"[SAVED] {fp_name}: CUMULATIVE saved ({total_pairs} pairs) -> {cum_path}")

match_all_floors_and_directions()

##### Visualization

Renders the matched floor-plan/elevation object pairs on top of the source image and saves the result to disk.


In [ ]:
import os
import json
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from matplotlib.patches import Polygon

def visualize_matched_objects(
    image_path,
    matched_json_path,
    output_path=None,
    use_bbox=False,
    show_id=True,
    show_category=True,
    show_center=True
):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with open(matched_json_path, "r", encoding="utf-8") as f:
        matched_data = json.load(f)

    if isinstance(matched_data, dict) and "annotations" in matched_data:
        matched_data = matched_data["annotations"]

    # Keep matched objects only
    matched_data = [obj for obj in matched_data if "matched" in obj and obj["matched"] is not None]

    if not isinstance(matched_data, list):
        raise ValueError("matched_json should be a list of object annotations")

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(img)

    def random_color():
        return [random.random(), random.random(), random.random()]

    for ann in matched_data:
        obj_id = ann.get("id","?")
        cat_name = ann.get("category_name","Unknown")
        bbox = ann.get("bbox", [])
        seg = ann.get("segmentation", [])
        center = ann.get("original_center", [None, None])
        color = random_color()

        if use_bbox:
            if len(bbox) == 4:
                x, y, w, h = bbox
                coords = np.array([[x, y],[x+w, y],[x+w, y+h],[x, y+h]])
                shape_patch = Polygon(coords, closed=True, edgecolor=color, facecolor='none', linewidth=1.5)
                ax.add_patch(shape_patch)
                label_str = ""
                if show_id: label_str += f"ID={obj_id} "
                if show_category: label_str += f"({cat_name})"
                ax.text(x, max(y-5, 0), label_str, color="yellow", fontsize=9)
        else:
            if isinstance(seg, list) and seg:
                if isinstance(seg[0], (int,float)):
                    seg = [seg]
                for poly_coords in seg:
                    coords = np.array(poly_coords).reshape(-1,2)
                    shape_patch = Polygon(coords, closed=True, edgecolor=color, facecolor='none', linewidth=1.0)
                    ax.add_patch(shape_patch)
                if seg and len(seg[0])>=2:
                    pts = np.array(seg[0]).reshape(-1,2)
                    x_label, y_label = pts[0]
                    label_str = ""
                    if show_id: label_str += f"ID={obj_id} "
                    if show_category: label_str += f"({cat_name})"
                    ax.text(x_label, max(y_label-5, 0), label_str, color="blue", fontsize=6)

        if show_center and center[0] is not None and center[1] is not None:
            cx, cy = center
            ax.scatter(cx, cy, s=6, c="red", marker="o")

    ax.axis("off")

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, bbox_inches="tight", dpi=600)
        plt.close(fig)
        print(f"[SAVED] matched visualization -> {output_path}")
    else:
        plt.show()

# ===============================
# Batch loop over the folder (ALL + Single)
# ===============================
dir_images   = "Drawings/FP/Images"           # 9 source images
dir_matchedA = "Drawings/matched/outputs"     # lowercase variant
dir_matchedB = "Drawings/Matched/outputs"     # uppercase variant
dir_output   = "Drawings/Matched/outputs/viz_Matched"

# Pick whichever matched-output root actually exists
if os.path.isdir(dir_matchedA):
    dir_matched_root = dir_matchedA
elif os.path.isdir(dir_matchedB):
    dir_matched_root = dir_matchedB
else:
    raise FileNotFoundError("Could not find a Matched/outputs directory.")

dir_all    = os.path.join(dir_matched_root, "ALL")
dir_single = os.path.join(dir_matched_root, "Single")

os.makedirs(dir_output, exist_ok=True)

orientations = ["North", "South", "East", "West"]  # used to identify Single files

image_files = [f for f in os.listdir(dir_images) if f.lower().endswith((".png",".jpg",".jpeg"))]
if not image_files:
    print(f"[WARNING] No images found in {dir_images}")

for img_file in image_files:
    base = os.path.splitext(img_file)[0]  # e.g. TEST_1
    image_path = os.path.join(dir_images, img_file)

    # Candidate JSONs: files starting with `base` in either ALL or Single
    candidates = []
    if os.path.isdir(dir_all):
        candidates += glob(os.path.join(dir_all, f"{base}*.json"))
    if os.path.isdir(dir_single):
        candidates += glob(os.path.join(dir_single, f"{base}*.json"))

    if not candidates:
        print(f"[SKIP] No matched json for base '{base}' in {dir_matched_root}/(ALL|Single)")
        continue

    for matched_json_path in candidates:
        json_file = os.path.basename(matched_json_path)
        json_lower = json_file.lower()

        # Tag: prefer ALL; otherwise use the direction name (+ "Single" if present in the filename)
        if "_all" in json_lower or json_lower.endswith("_all.json"):
            orient_tag = "ALL"
        else:
            found_orient = None
            for ori in orientations:
                if ori.lower() in json_lower:
                    found_orient = ori
                    break
            if "single" in json_lower and found_orient:
                orient_tag = f"{found_orient}"
            elif found_orient:
                orient_tag = found_orient
            else:
                orient_tag = "Matched"

        output_path = os.path.join(dir_output, f"{base}_{orient_tag}.png")

        print(f"\n[PROCESSING] image={img_file}, json={json_file} -> {orient_tag}")
        visualize_matched_objects(
            image_path=image_path,
            matched_json_path=matched_json_path,
            output_path=output_path,
            use_bbox=False,   # set True to draw bounding boxes instead of segmentation
            show_id=True,
            show_category=True,
            show_center=True
        )

#### 3.2. Update matching result (Final version)

In [ ]:
import os
import json

def update_width_depth(json_path, output_path):
    """Normalize Window object_width/object_depth so width is always the larger value."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    target_categories = ["Window"]

    for ann in data["annotations"]:
        if ann["category_name"] in target_categories:
            width = ann["object_width"]
            depth = ann["object_depth"]
            ann["object_width"], ann["object_depth"] = max(width, depth), min(width, depth)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"[SAVED] {output_path}")

# Batch loop over the folder
input_dir = "Drawings/Matched/outputs/ALL"   # existing JSON folder
output_dir = "Drawings/Matched/Final"        # output folder

json_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".json")]

if not json_files:
    print("[WARNING] No JSON files found in", input_dir)

for json_file in json_files:
    input_path = os.path.join(input_dir, json_file)
    output_path = os.path.join(output_dir, json_file.replace(".json", "_Final.json"))

    update_width_depth(input_path, output_path)

# 4. Post-processing

Converts the matched detections into wall centerlines, corner points, and hosted openings (windows/doors snapped to their wall) needed to build BIM geometry.

- Converting to BIM data ultimately requires start and end points.
- Handling this via segmentation masks or bounding boxes is impractical due to the highly irregular nature of the data, necessitating a conversion step to generate BIM data.


#### 4.1. Skeletonization

In [ ]:
import os
import json
import numpy as np
import cv2
from glob import glob

def load_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(json_data, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=4)
    print(f"[SAVED] Updated JSON saved to {output_path}")

def create_binary_mask(segmentation, img_size):
    mask = np.zeros((img_size[1], img_size[0]), dtype=np.uint8)
    for segment in segmentation:
        pts = np.array(segment, dtype=np.int32).reshape(-1, 2)
        cv2.fillPoly(mask, [pts], 255)
    return mask

# =============================
# Zhang-Suen thinning (NumPy)
# =============================
def _zs_thinning(binary_img_255):
    """
    Zhang-Suen thinning, returning a 0/255 image. Equivalent to
    OpenCV's ximgproc.thinning(..., THINNING_ZHANGSUEN).
    binary_img_255: uint8 image with values in {0, 255}
    """
    img = (binary_img_255 > 0).astype(np.uint8)

    changed = True
    while changed:
        changed = False

        # Shift to get each of the 8 neighbors (P2..P9)
        P2 = np.pad(img[0:-1, :   ], ((1,0),(0,0)))  # up
        P3 = np.pad(img[0:-1, 1:  ], ((1,0),(0,1)))  # up-right
        P4 = np.pad(img[:,   1:  ], ((0,0),(0,1)))   # right
        P5 = np.pad(img[1:,  1:  ], ((0,1),(0,1)))   # down-right
        P6 = np.pad(img[1:,  :   ], ((0,1),(0,0)))   # down
        P7 = np.pad(img[1:,  0:-1], ((0,1),(1,0)))   # down-left
        P8 = np.pad(img[:,   0:-1], ((0,0),(1,0)))   # left
        P9 = np.pad(img[0:-1,0:-1], ((1,0),(1,0)))   # up-left

        # Neighbor sum B(p)
        B = P2+P3+P4+P5+P6+P7+P8+P9

        # Count of 0->1 transitions A(p), in order P2,P3,...,P9,P2
        seq = [P2,P3,P4,P5,P6,P7,P8,P9,P2]
        transitions = np.zeros_like(img, dtype=np.uint8)
        for k in range(8):
            transitions += ((seq[k] == 0) & (seq[k+1] == 1)).astype(np.uint8)
        A = transitions

        # Step 1 conditions
        C1 = (img == 1)
        C2 = (B >= 2) & (B <= 6)
        C3 = (A == 1)
        C4 = (P2 * P4 * P6 == 0)
        C5 = (P4 * P6 * P8 == 0)
        m1 = C1 & C2 & C3 & C4 & C5

        if np.any(m1):
            img = img & (~m1).astype(np.uint8)
            changed = True

        # Recompute neighbors
        P2 = np.pad(img[0:-1, :   ], ((1,0),(0,0)))
        P3 = np.pad(img[0:-1, 1:  ], ((1,0),(0,1)))
        P4 = np.pad(img[:,   1:  ], ((0,0),(0,1)))
        P5 = np.pad(img[1:,  1:  ], ((0,1),(0,1)))
        P6 = np.pad(img[1:,  :   ], ((0,1),(0,0)))
        P7 = np.pad(img[1:,  0:-1], ((0,1),(1,0)))
        P8 = np.pad(img[:,   0:-1], ((0,0),(1,0)))
        P9 = np.pad(img[0:-1,0:-1], ((1,0),(1,0)))

        B = P2+P3+P4+P5+P6+P7+P8+P9

        seq = [P2,P3,P4,P5,P6,P7,P8,P9,P2]
        transitions = np.zeros_like(img, dtype=np.uint8)
        for k in range(8):
            transitions += ((seq[k] == 0) & (seq[k+1] == 1)).astype(np.uint8)
        A = transitions

        # Step 2 conditions
        C1 = (img == 1)
        C2 = (B >= 2) & (B <= 6)
        C3 = (A == 1)
        C4 = (P2 * P4 * P8 == 0)
        C5 = (P2 * P6 * P8 == 0)
        m2 = C1 & C2 & C3 & C4 & C5

        if np.any(m2):
            img = img & (~m2).astype(np.uint8)
            changed = True

    return (img * 255).astype(np.uint8)

def extract_skeleton_opencv(mask):
    # Name kept for compatibility; the implementation is Zhang-Suen thinning
    return _zs_thinning(mask)

def process_json(json_data, img_size):
    for ann in json_data["annotations"]:
        # Wall objects only (category_id == 2)
        if ann.get("category_id") not in [2]:
            continue
        seg = ann.get("segmentation", [])
        if not seg:
            continue

        mask = create_binary_mask(seg, img_size)
        skeleton = extract_skeleton_opencv(mask)

        # (y, x) -> (x, y)
        coords = np.column_stack(np.where(skeleton > 0))[:, ::-1]
        ann["skeleton"] = coords.tolist()
    return json_data

def run_folder(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    json_files = glob(os.path.join(input_dir, "*.json"))
    if not json_files:
        print(f"[WARNING] No JSON files found in {input_dir}")
        return

    print("\n=== Skeleton Generation Start ===")
    for json_path in json_files:
        base = os.path.basename(json_path)
        output_path = os.path.join(output_dir, base)
        try:
            json_data = load_json(json_path)
            if "images" in json_data and json_data["images"]:
                image_info = json_data["images"][0]
                img_size = (image_info["width"], image_info["height"])
            else:
                print(f"[WARNING] No image info -> {json_path}")
                continue

            updated_json = process_json(json_data, img_size)
            save_json(updated_json, output_path)
        except Exception as e:
            print(f"[ERROR] Error processing {json_path}: {e}")

    print("\n[DONE] Skeletonization completed for all files!")

input_dir = "Drawings/Matched/Final"            # input folder (ALL *.json)
output_dir = "Drawings/PostProcessing/Skeleton" # output folder
run_folder(input_dir, output_dir)

##### 4.1.1. Skeleton coordinate simplification

In [ ]:
import os
import json
from glob import glob

def round_to_nearest_30(n):
    return round(n / 30) * 30

def load_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(json_data, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=4)
    print(f"[SAVED] Updated JSON saved to {output_path}")

def process_one_file(json_path):
    a = load_json(json_path)

    # 1) Round skeleton coordinates to the nearest 30 and drop duplicates (globally)
    seen_points = []
    for i in a["annotations"]:
        if i.get("category_name") in ["Door", "Window", "Slab"]:
            continue
        skeleton = i.get("skeleton", [])
        rounded = []
        for j in range(len(skeleton)):
            c = [round_to_nearest_30(skeleton[j][0]),
                 round_to_nearest_30(skeleton[j][1])]
            if c in seen_points:
                continue
            else:
                seen_points.append(c)
            rounded.append(c)
        i["skeleton"] = rounded

    # 2) Tally how often each x/y coordinate occurs
    x_counts = {}
    y_counts = {}
    for i in a['annotations']:
        if i.get("category_name") in ["Door", "Window", "Slab"]:
            continue
        skeleton = i.get("skeleton", [])
        for s in skeleton:
            x_, y_ = s[0], s[1]
            x_counts[x_] = x_counts.get(x_, 0) + 1
            y_counts[y_] = y_counts.get(y_, 0) + 1

    # 3) Keep only coordinates that occur at least 10 times
    x_frequent = {kk: vv for kk, vv in x_counts.items() if vv >= 10}
    y_frequent = {kk: vv for kk, vv in y_counts.items() if vv >= 10}

    # 4) Keep only points on a frequent x or y line
    for i in a['annotations']:
        if i.get("category_name") in ["Door", "Window", "Slab"]:
            continue
        skeleton = i.get("skeleton", [])
        new_skeleton = []
        for s in skeleton:
            x_, y_ = s[0], s[1]
            if x_ in x_frequent or y_ in y_frequent:
                new_skeleton.append(s)
        i["skeleton"] = new_skeleton

    return a

def run_folder(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = glob(os.path.join(input_dir, "*.json"))
    if not files:
        print(f"[WARNING] No JSON files found in {input_dir}")
        return

    print("\n=== Skeleton post-filtering start ===")
    for json_path in sorted(files):
        try:
            base = os.path.basename(json_path)
            out_path = os.path.join(output_dir, base)
            updated = process_one_file(json_path)
            save_json(updated, out_path)
        except Exception as e:
            print(f"[ERROR] Error processing {json_path}: {e}")
    print("[DONE] Completed for all files.")

# ==== Example run ====
input_dir = "Drawings/PostProcessing/Skeleton"      # input folder
output_dir = "Drawings/PostProcessing/Skeleton2"    # output folder
run_folder(input_dir, output_dir)

##### 4.1.2. Skeleton-based wall line vectorization

In [ ]:
import os
import json
import numpy as np
from glob import glob

def distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def normalize_skeleton_data(skeleton_data):
    if isinstance(skeleton_data, list) and all(isinstance(p, list) and len(p) == 2 for p in skeleton_data):
        return skeleton_data
    elif isinstance(skeleton_data, list) and all(isinstance(v, (int, float)) for v in skeleton_data):
        return [[skeleton_data[i], skeleton_data[i + 1]] for i in range(0, len(skeleton_data), 2)]
    return []

def merge_connected_lines(skeleton_points, threshold=30):
    skeleton_points = sorted(set(tuple(p) for p in skeleton_points))  # remove duplicates
    lines = []

    # horizontal group
    y_groups = {}
    for x, y in skeleton_points:
        y_groups.setdefault(y, []).append((x, y))

    for y, pts in y_groups.items():
        pts.sort()
        line = [pts[0]]
        for i in range(1, len(pts)):
            if pts[i][0] - pts[i-1][0] <= threshold:
                line.append(pts[i])
            else:
                if len(line) >= 2:
                    lines.append({
                        "start_point": line[0],
                        "end_point": line[-1],
                        "direction": "H",
                        "skeleton": line.copy()
                    })
                line = [pts[i]]
        if len(line) >= 2:
            lines.append({
                "start_point": line[0],
                "end_point": line[-1],
                "direction": "H",
                "skeleton": line.copy()
            })

    # vertical group
    x_groups = {}
    for x, y in skeleton_points:
        x_groups.setdefault(x, []).append((x, y))

    for x, pts in x_groups.items():
        pts.sort(key=lambda p: p[1])
        line = [pts[0]]
        for i in range(1, len(pts)):
            if pts[i][1] - pts[i-1][1] <= threshold:
                line.append(pts[i])
            else:
                if len(line) >= 2:
                    lines.append({
                        "start_point": line[0],
                        "end_point": line[-1],
                        "direction": "V",
                        "skeleton": line.copy()
                    })
                line = [pts[i]]
        if len(line) >= 2:
            lines.append({
                "start_point": line[0],
                "end_point": line[-1],
                "direction": "V",
                "skeleton": line.copy()
            })

    return lines

def process_wall_annotations_global(input_json_path, output_json_path, threshold=30):
    with open(input_json_path, "r") as f:
        data = json.load(f)

    all_annotations = data["annotations"]
    updated_annotations = []
    new_wall_annotations = []

    next_id = max(ann["id"] for ann in all_annotations) + 1

    # 1. Pool every Wall object's skeleton points into one global set
    all_wall_skeleton_points = []
    wall_original_annotations = []

    for ann in all_annotations:
        if ann.get("category_name") == "Wall":
            skeleton = normalize_skeleton_data(ann.get("skeleton", []))
            all_wall_skeleton_points.extend(skeleton)

            # Keep the original Wall object, relabeled so it isn't confused with the new ones
            ann["category_name"] = "WallOriginal"
            ann["category_id"] = 3
            wall_original_annotations.append(ann)
        else:
            # Non-wall objects pass through unchanged
            updated_annotations.append(ann)

    # 2. Build straight line segments from the pooled skeleton points
    line_segments = merge_connected_lines(all_wall_skeleton_points, threshold=threshold)

    for line in line_segments:
        new_wall_annotations.append({
            "id": next_id,
            "image_id": wall_original_annotations[0]["image_id"] if wall_original_annotations else None,
            "category_id": 2,
            "category_name": "Wall",
            "skeleton": line["skeleton"],
            "start_point": line["start_point"],
            "end_point": line["end_point"],
            "orientation": line["direction"]
        })
        next_id += 1

    # 3. Replace annotations with WallOriginal + untouched objects + new Wall segments
    data["annotations"] = updated_annotations + wall_original_annotations + new_wall_annotations

    with open(output_json_path, "w") as f:
        json.dump(data, f, indent=4)

    print(f"[SAVED] Processed: '{output_json_path}'")

def run_folder(input_dir, output_dir, threshold=30):
    os.makedirs(output_dir, exist_ok=True)
    files = sorted(glob(os.path.join(input_dir, "*.json")))
    if not files:
        print(f"[WARNING] No JSON files found in {input_dir}")
        return

    print(f"\n=== Wall skeleton -> line segments (global merge) start ===")
    print(f"Input:  {input_dir}")
    print(f"Output: {output_dir}")
    print(f"Threshold: {threshold}")

    for idx, in_path in enumerate(files, 1):
        base = os.path.basename(in_path)
        out_path = os.path.join(output_dir, base)
        try:
            process_wall_annotations_global(in_path, out_path, threshold=threshold)
        except Exception as e:
            print(f"[ERROR] [{idx}/{len(files)}] {in_path}: {e}")
    print("[DONE] Completed for all files.")

# ==== Example run ====
input_dir  = "Drawings/PostProcessing/Skeleton2"   # input folder
output_dir = "Drawings/PostProcessing/Skeleton3"   # output folder
run_folder(input_dir, output_dir, threshold=30)

#### 4.2. defining start point, end point, and direction

In [ ]:
import os
import json
from glob import glob

def update_wall_skeleton_with_connections(json_path, output_path, threshold=100):
    with open(json_path, "r") as f:
        data = json.load(f)

    # 1. Collect Wall objects
    lines = []
    wall_ann_map = {}

    for ann in data["annotations"]:
        if ann.get("category_name") == "Wall":
            start = ann.get("start_point")
            end = ann.get("end_point")
            direction = ann.get("orientation")
            if start and end and direction:
                line = {
                    "id": ann["id"],
                    "start_point": tuple(start),
                    "end_point": tuple(end),
                    "direction": direction
                }
                lines.append(line)
                wall_ann_map[ann["id"]] = ann

    # 2. Determine whether wall endpoints touch
    def is_point_on_line(point, line, threshold=5):
        x, y = point
        sx, sy = line["start_point"]
        ex, ey = line["end_point"]

        if line["direction"] == "H":
            return abs(sy - y) <= threshold and min(sx, ex) <= x <= max(sx, ex)
        elif line["direction"] == "V":
            return abs(sx - x) <= threshold and min(sy, ey) <= y <= max(sy, ey)
        return False

    def check_contact_points(lines, threshold=5):
        results = []
        for line in lines:
            start_contact = any(
                is_point_on_line(line["start_point"], other_line, threshold)
                for other_line in lines if other_line["id"] != line["id"]
            )
            end_contact = any(
                is_point_on_line(line["end_point"], other_line, threshold)
                for other_line in lines if other_line["id"] != line["id"]
            )
            results.append({
                "id": line["id"],
                "start_point": line["start_point"],
                "start_contact": start_contact,
                "end_point": line["end_point"],
                "end_contact": end_contact,
                "direction": line["direction"]
            })
        return results

    # 3. Find the nearest unconnected point
    def find_nearest_unconnected(point, candidates, threshold):
        x, y = point
        nearest = None
        min_dist = float('inf')
        for other_point in candidates:
            if point == other_point:
                continue
            ox, oy = other_point
            if abs(x - ox) <= threshold and abs(y - oy) <= threshold:
                dist = ((x - ox) ** 2 + (y - oy) ** 2) ** 0.5
                if dist < min_dist:
                    min_dist = dist
                    nearest = other_point
        return nearest

    contact_results = check_contact_points(lines, threshold=5)
    unconnected_points = [
        (res["start_point"], res["direction"], res["id"], "start") for res in contact_results if not res["start_contact"]
    ] + [
        (res["end_point"], res["direction"], res["id"], "end") for res in contact_results if not res["end_contact"]
    ]

    for point, direction, wall_id, point_type in unconnected_points:
        candidates = [pt for pt, _, _, _ in unconnected_points]
        nearest = find_nearest_unconnected(point, candidates, threshold)

        if nearest:
            px, py = point
            nx, ny = nearest
            match = [(d, wid, pt_type) for pt, d, wid, pt_type in unconnected_points if pt == nearest]
            if not match:
                continue
            nearest_direction, nearest_id, nearest_pt_type = match[0]

            if direction == nearest_direction:
                if direction == "H" and py == ny:
                    if point_type == "start":
                        wall_ann_map[wall_id]["start_point"] = [min(px, nx), py]
                    else:
                        wall_ann_map[wall_id]["end_point"] = [max(px, nx), py]
                elif direction == "V" and px == nx:
                    if point_type == "start":
                        wall_ann_map[wall_id]["start_point"] = [px, min(py, ny)]
                    else:
                        wall_ann_map[wall_id]["end_point"] = [px, max(py, ny)]
            else:
                cross_point = [px, ny] if direction == "V" else [nx, py]

                if point_type == "start":
                    wall_ann_map[wall_id]["start_point"] = cross_point
                else:
                    wall_ann_map[wall_id]["end_point"] = cross_point

                if nearest_pt_type == "start":
                    wall_ann_map[nearest_id]["start_point"] = cross_point
                else:
                    wall_ann_map[nearest_id]["end_point"] = cross_point

                for wid in [wall_id, nearest_id]:
                    wall = wall_ann_map[wid]
                    if "skeleton" not in wall:
                        wall["skeleton"] = []
                    if cross_point not in wall["skeleton"]:
                        wall["skeleton"].append(cross_point)

    with open(output_path, "w") as f:
        json.dump(data, f, indent=4)

    print(f"[SAVED] Updated: {output_path}")


# Batch loop over the folder (logic unchanged)
input_dir = "Drawings/PostProcessing/Skeleton3"
output_dir = "Drawings/PostProcessing/Skeleton4"
os.makedirs(output_dir, exist_ok=True)

json_files = glob(os.path.join(input_dir, "*.json"))

if not json_files:
    print("[WARNING] No JSON files found!")
else:
    print("\n=== Processing ===")

for json_path in json_files:
    base = os.path.basename(json_path)
    output_path = os.path.join(output_dir, base)

    update_wall_skeleton_with_connections(json_path, output_path, threshold=100)

print("\n[DONE] All files updated!")

##### Visualization

Renders the extracted wall centerlines and corner points and saves the result to disk.


In [ ]:
# Wall center lines & corner points visualization (OpenCV-only)
import os
import json
import cv2
import numpy as np
from glob import glob

def load_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def draw_walls_and_corners(json_path, output_path):
    data = load_json(json_path)
    anns = data.get("annotations", [])

    # Image size (defaults to 2048x2048 if not present)
    if "images" in data and data["images"]:
        w = int(data["images"][0].get("width", 2048))
        h = int(data["images"][0].get("height", 2048))
    else:
        w, h = 2048, 2048

    canvas = np.ones((h, w, 3), dtype=np.uint8) * 255   # white background (BGR)

    # Collect Wall objects only
    walls = []
    for ann in anns:
        if ann.get("category_name") == "Wall":
            sp = ann.get("start_point")
            ep = ann.get("end_point")
            ori = ann.get("orientation")
            if sp and ep and ori:
                walls.append({
                    "id": ann.get("id"),
                    "start": (int(sp[0]), int(sp[1])),
                    "end": (int(ep[0]), int(ep[1])),
                    "ori": ori
                })

    # Draw wall center lines (black)
    for wline in walls:
        cv2.line(canvas, wline["start"], wline["end"], (0, 0, 0), 2, lineType=cv2.LINE_AA)

    # Find corner points (H/V line intersections)
    corner_points = set()
    for i in range(len(walls)):
        w1 = walls[i]
        for j in range(len(walls)):
            if i == j:
                continue
            w2 = walls[j]

            if w1["ori"] == "H" and w2["ori"] == "V":
                hy = w1["start"][1]
                vx = w2["start"][0]

                if (min(w1["start"][0], w1["end"][0]) <= vx <= max(w1["start"][0], w1["end"][0])
                    and min(w2["start"][1], w2["end"][1]) <= hy <= max(w2["start"][1], w2["end"][1])):
                    corner_points.add((vx, hy))

            elif w1["ori"] == "V" and w2["ori"] == "H":
                vx = w1["start"][0]
                hy = w2["start"][1]

                if (min(w1["start"][1], w1["end"][1]) <= hy <= max(w1["start"][1], w1["end"][1])
                    and min(w2["start"][0], w2["end"][0]) <= vx <= max(w2["start"][0], w2["end"][0])):
                    corner_points.add((vx, hy))

    # Mark corners with a red circle
    for (cx, cy) in corner_points:
        cv2.circle(canvas, (cx, cy), 6, (0, 0, 255), -1, lineType=cv2.LINE_AA)

    ensure_dir(os.path.dirname(output_path))
    cv2.imwrite(output_path, canvas)
    print(f"[SAVED] {output_path}")

# ====== Batch-process the whole folder ======
input_dir  = "Drawings/PostProcessing/Skeleton4"
output_dir = "Drawings/PostProcessing/viz_Skeleton4"
ensure_dir(output_dir)

json_files = glob(os.path.join(input_dir, "*.json"))
print("\n=== Visualization start ===")

for json_path in json_files:
    base_png = os.path.splitext(os.path.basename(json_path))[0] + ".png"
    out_path = os.path.join(output_dir, base_png)
    draw_walls_and_corners(json_path, out_path)

print("\n[DONE] All visualizations completed and saved as PNG.")

#### 4.3. Defining window orientation, width, depth

In [ ]:
import os
import json
from glob import glob

def compute_window_points(annotation):
    bbox = annotation.get("bbox")
    if bbox is None or len(bbox) < 4:
        return None, None, None
    x_min, y_min, width, height = bbox
    x_max = x_min + width
    y_max = y_min + height

    if width > height:
        # Wide (horizontal) window
        start_point = [x_min, (y_min + y_max) / 2]
        end_point = [x_max, (y_min + y_max) / 2]
        orientation = "H"
    else:
        # Tall (vertical) window
        start_point = [(x_min + x_max) / 2, y_min]
        end_point = [(x_min + x_max) / 2, y_max]
        orientation = "V"

    return start_point, end_point, orientation


def update_windows_in_json(input_json_path, output_json_path):
    with open(input_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for ann in data.get("annotations", []):
        if ann.get("category_name") == "Window":
            sp, ep, orientation = compute_window_points(ann)
            ann["start_point"] = sp
            ann["end_point"] = ep
            ann["orientation"] = orientation

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"[SAVED] {output_json_path}")


# Batch-process the whole folder
input_dir = "Drawings/PostProcessing/Skeleton4"
output_dir = "Drawings/PostProcessing/Skeleton5"
os.makedirs(output_dir, exist_ok=True)

json_files = glob(os.path.join(input_dir, "*.json"))

print(f"\n=== Processing {len(json_files)} JSON files ===")

for json_path in json_files:
    base = os.path.basename(json_path)
    output_path = os.path.join(output_dir, base)
    update_windows_in_json(json_path, output_path)

print("\n[DONE] All files processed and saved.")

#### 4.4. Hosting opening objects(windows, doors) to walls 

In [ ]:
import os
import json
from glob import glob

def find_nearest_wall(window, walls):
    wx_start, wy_start = window["start_point"]
    wx_end, wy_end = window["end_point"]
    orientation = window["orientation"]

    min_distance = float('inf')
    nearest_wall = None

    for wall in walls:
        wx_center = (wx_start + wx_end) / 2
        wy_center = (wy_start + wy_end) / 2

        wall_start = wall["start_point"]
        wall_end = wall["end_point"]
        wall_orientation = wall["orientation"]

        if orientation == "H" and wall_orientation == "H":
            wall_y = wall_start[1]
            distance = abs(wy_center - wall_y)
            if distance < min_distance:
                min_distance = distance
                nearest_wall = wall

        elif orientation == "V" and wall_orientation == "V":
            wall_x = wall_start[0]
            distance = abs(wx_center - wall_x)
            if distance < min_distance:
                min_distance = distance
                nearest_wall = wall

    return nearest_wall


def update_windows_with_hosted_walls(json_path, output_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    walls = [ann for ann in data["annotations"] if ann.get("category_name") == "Wall"]
    windows = [ann for ann in data["annotations"] if ann.get("category_name") == "Window"]

    for window in windows:
        nearest_wall = find_nearest_wall(window, walls)
        if nearest_wall:
            wall_id = nearest_wall["id"]
            window["hosted"] = wall_id

            if window["orientation"] == "H":
                wall_y = nearest_wall["start_point"][1]
                window["start_point"][1] = wall_y
                window["end_point"][1] = wall_y
                window["bbox"][1] = wall_y

            elif window["orientation"] == "V":
                wall_x = nearest_wall["start_point"][0]
                window["start_point"][0] = wall_x
                window["end_point"][0] = wall_x
                window["bbox"][0] = wall_x

            new_x_center = window["bbox"][0] + (window["bbox"][2] / 2)
            new_y_center = window["bbox"][1] + (window["bbox"][3] / 2)

            original_z = window["transformed_center"][2] if "transformed_center" in window else 1
            window["original_center"] = [new_x_center, new_y_center]
            window["transformed_center"] = [new_x_center, new_y_center, original_z]

    with open(output_path, "w") as f:
        json.dump(data, f, indent=4)

    print(f"[SAVED] {output_path}")


# Batch loop over the folder
input_dir = "Drawings/PostProcessing/Skeleton5"
output_dir = "Drawings/PostProcessing/Skeleton6"
os.makedirs(output_dir, exist_ok=True)

json_files = glob(os.path.join(input_dir, "*.json"))

print(f"\n=== Hosting windows ({len(json_files)} files) ===")

for json_path in json_files:
    base_name = os.path.basename(json_path)
    output_path = os.path.join(output_dir, base_name)
    update_windows_with_hosted_walls(json_path, output_path)

print("\n[DONE] All files processed and saved.")

In [ ]:
import os
import json
import math
from glob import glob

def distance(point1, point2):
    return math.sqrt((point1[0] - point2[0]) ** 2 + (point1[1] - point2[1]) ** 2)

def is_wall_intersecting_bbox(wall, bbox):
    x_min, y_min, width, height = bbox
    x_max = x_min + width
    y_max = y_min + height

    wx1, wy1 = wall["start_point"]
    wx2, wy2 = wall["end_point"]

    if wall["orientation"] == "H":
        return y_min <= wy1 <= y_max and not (wx2 < x_min or wx1 > x_max)
    elif wall["orientation"] == "V":
        return x_min <= wx1 <= x_max and not (wy2 < y_min or wx1 > x_max)
    return False

def find_nearest_wall(door, walls, orientation):
    min_dist = float("inf")
    nearest_wall = None
    door_center = door.get("original_center", door.get("transformed_center", [0, 0]))

    for wall in walls:
        if wall["orientation"] == orientation:
            wall_center = [
                (wall["start_point"][0] + wall["end_point"][0]) / 2,
                (wall["start_point"][1] + wall["end_point"][1]) / 2
            ]
            dist = distance(door_center, wall_center)
            if dist < min_dist:
                min_dist = dist
                nearest_wall = wall

    return nearest_wall

def update_doors_with_walls(json_path, output_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    walls = [ann for ann in data["annotations"] if ann.get("category_name") == "Wall"]
    doors = [ann for ann in data["annotations"] if ann.get("category_name") == "Door"]

    processed_doors = set()

    # Condition 1: door bbox intersects a wall band -> snap to that wall
    for door in doors:
        bbox = door.get("bbox")
        if not bbox:
            continue

        for wall in walls:
            if is_wall_intersecting_bbox(wall, bbox):
                door["hosted"] = wall["id"]
                width, height = bbox[2], bbox[3]
                x_center = bbox[0] + width / 2
                y_center = bbox[1] + height / 2

                if wall["orientation"] == "H":
                    if y_center < wall["start_point"][1]:
                        new_y_min = wall["start_point"][1] - height
                    else:
                        new_y_min = wall["start_point"][1]
                    bbox = [bbox[0], new_y_min, width, height]
                    start_point = [bbox[0], new_y_min + height / 2]
                    end_point = [bbox[0] + width, new_y_min + height / 2]

                elif wall["orientation"] == "V":
                    if x_center < wall["start_point"][0]:
                        new_x_min = wall["start_point"][0] - width
                    else:
                        new_x_min = wall["start_point"][0]
                    bbox = [new_x_min, bbox[1], width, height]
                    start_point = [new_x_min + width / 2, bbox[1]]
                    end_point = [new_x_min + width / 2, bbox[1] + height]

                new_x_center = bbox[0] + width / 2
                new_y_center = bbox[1] + height / 2
                z = door["transformed_center"][2] if "transformed_center" in door else 1

                door["start_point"] = start_point
                door["end_point"] = end_point
                door["orientation"] = wall["orientation"]
                door["bbox"] = bbox
                door["original_center"] = [new_x_center, new_y_center]
                door["transformed_center"] = [new_x_center, new_y_center, z]

                processed_doors.add(door["id"])
                break

    # Condition 2: otherwise, snap to the nearest wall with the same orientation
    for door in doors:
        if door["id"] in processed_doors:
            continue

        bbox = door.get("bbox")
        if not bbox:
            continue

        x_min, y_min, width, height = bbox
        x_max = x_min + width
        y_max = y_min + height
        x_center = x_min + width / 2
        y_center = y_min + height / 2

        door_orientation = "V" if width < height else "H"
        candidate_walls = [w for w in walls if w["orientation"] == door_orientation]
        best_wall = find_nearest_wall(door, candidate_walls, door_orientation)

        if best_wall:
            door["hosted"] = best_wall["id"]

            if door_orientation == "H":
                wall_y = best_wall["start_point"][1]
                new_y_min = wall_y - height if y_center < wall_y else wall_y
                bbox = [x_min, new_y_min, width, height]
                start_point = [x_min, new_y_min + height / 2]
                end_point = [x_min + width, new_y_min + height / 2]

            elif door_orientation == "V":
                wall_x = best_wall["start_point"][0]
                new_x_min = wall_x - width if x_center < wall_x else wall_x
                bbox = [new_x_min, y_min, width, height]
                start_point = [new_x_min + width / 2, y_min]
                end_point = [new_x_min + width / 2, y_min + height]

            new_x_center = bbox[0] + width / 2
            new_y_center = bbox[1] + height / 2
            z = door["transformed_center"][2] if "transformed_center" in door else 1

            door["start_point"] = start_point
            door["end_point"] = end_point
            door["orientation"] = door_orientation
            door["bbox"] = bbox
            door["original_center"] = [new_x_center, new_y_center]
            door["transformed_center"] = [new_x_center, new_y_center, z]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"[SAVED] Door hosting & alignment done. Saved: {output_path}")


# ===== Batch loop: Skeleton6 -> Skeleton7 =====
input_dir = "Drawings/PostProcessing/Skeleton6"
output_dir = "Drawings/PostProcessing/Skeleton7"
os.makedirs(output_dir, exist_ok=True)

json_files = glob(os.path.join(input_dir, "*.json"))
print(f"\n=== Processing {len(json_files)} files ===")

if not json_files:
    print("[WARNING] No JSON files found.")
else:
    for src in json_files:
        dst = os.path.join(output_dir, os.path.basename(src))
        try:
            update_doors_with_walls(src, dst)
        except Exception as e:
            print(f"[ERROR] {src} -> {e}")

print("\n[DONE] All files processed and saved.")

In [ ]:
import os
import json
import math
from glob import glob

def distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def is_wall_intersecting_bbox(wall, bbox):
    x_min, y_min, width, height = bbox
    x_max = x_min + width
    y_max = y_min + height

    wx1, wy1 = wall["start_point"]
    wx2, wy2 = wall["end_point"]

    if wall["orientation"] == "H":
        return y_min <= wy1 <= y_max and not (wx2 < x_min or wx1 > x_max)
    elif wall["orientation"] == "V":
        return x_min <= wx1 <= x_max and not (wy2 < y_min or wx1 > y_max)
    return False

def find_nearest_wall(door_center, candidate_walls):
    min_dist = float("inf")
    nearest_wall = None
    for wall in candidate_walls:
        wall_center = [
            (wall["start_point"][0] + wall["end_point"][0]) / 2,
            (wall["start_point"][1] + wall["end_point"][1]) / 2
        ]
        dist = distance(door_center, wall_center)
        if dist < min_dist:
            min_dist = dist
            nearest_wall = wall
    return nearest_wall

def update_doors_with_improved_logic(json_path, output_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    walls = [ann for ann in data["annotations"] if ann.get("category_name") == "Wall"]
    doors = [ann for ann in data["annotations"] if ann.get("category_name") == "Door"]

    for door in doors:
        bbox = door.get("bbox")
        if not bbox:
            continue

        x_min, y_min, width, height = bbox
        x_center = x_min + width / 2
        y_center = y_min + height / 2

        # 1) set door orientation
        if width > height:
            door_orientation = "V"
        else:
            door_orientation = "H"
        door["orientation"] = door_orientation

        # 2) filter walls by orientation
        candidate_walls = [w for w in walls if w["orientation"] == door_orientation]

        # 3) prefer an intersecting wall
        intersecting_wall = None
        for wall in candidate_walls:
            if is_wall_intersecting_bbox(wall, bbox):
                intersecting_wall = wall
                break

        # 4) otherwise fall back to the nearest wall
        if not intersecting_wall:
            intersecting_wall = find_nearest_wall([x_center, y_center], candidate_walls)
        if not intersecting_wall:
            continue

        door["hosted"] = intersecting_wall["id"]
        wall_y = intersecting_wall["start_point"][1]
        wall_x = intersecting_wall["start_point"][0]

        # 5) adjust bbox, start/end points, and centers
        if door_orientation == "H":
            if y_center < wall_y:
                new_y_min = wall_y - height
            else:
                new_y_min = wall_y
            bbox = [x_min, new_y_min, width, height]
            start_point = [x_min, new_y_min + height / 2]
            end_point = [x_min + width, new_y_min + height / 2]

        elif door_orientation == "V":
            if x_center < wall_x:
                new_x_min = wall_x - width
            else:
                new_x_min = wall_x
            bbox = [new_x_min, y_min, width, height]
            start_point = [new_x_min + width / 2, y_min]
            end_point = [new_x_min + width / 2, y_min + height]

        new_x_center = bbox[0] + width / 2
        new_y_center = bbox[1] + height / 2
        z = door.get("transformed_center", [0, 0, 1])[2]

        door["bbox"] = bbox
        door["start_point"] = start_point
        door["end_point"] = end_point
        door["original_center"] = [new_x_center, new_y_center]
        door["transformed_center"] = [new_x_center, new_y_center, z]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)
    print(f"[SAVED] Doors updated -> {output_path}")

# ===== Folder loop: Skeleton7 -> Skeleton8 =====
input_dir = "Drawings/PostProcessing/Skeleton7"
output_dir = "Drawings/PostProcessing/Skeleton8"
os.makedirs(output_dir, exist_ok=True)

files = glob(os.path.join(input_dir, "*.json"))
print(f"\n=== Processing {len(files)} files ===")

for src in files:
    dst = os.path.join(output_dir, os.path.basename(src))
    try:
        update_doors_with_improved_logic(src, dst)
    except Exception as e:
        print(f"[ERROR] Failed on {src}: {e}")

print("\n[DONE] All files processed.")

# 5. Dimensional information recognition & calibration

Recovers the real-world drawing scale from paper size + scale ratio (via GPT-4o-mini) and uses it to convert every coordinate from pixels to millimeters.


#### 5.1. LMM-based dimensional information recognition(scale, height)

In [ ]:
import os
import json
import base64
import re
from openai import OpenAI
from PIL import Image
from io import BytesIO

# Quiet down the HTTP client logging from the OpenAI SDK
import logging
for _name in ("httpx", "httpcore", "openai"):
    logging.getLogger(_name).setLevel(logging.WARNING)
# To silence other INFO-level logs too, uncomment:
# logging.basicConfig(level=logging.WARNING, force=True)

# Requires the OPENAI_API_KEY environment variable to be set.
openai_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_key)

def encode_image_object(image):
    """Convert a PIL image object to a base64-encoded string."""
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def request_gpt_fix(scale_text):
    """Ask the model to correct its own output if it doesn't match the expected format."""
    prompt = f"""
    The scale information you provided may be in the wrong format.
    The correct format must always be `<paper size>: <scale>`.

    Correct examples
    - "A1: 1/40"
    - "A3: 1/80"
    - "A2: 1/100"

    Incorrect formats
    - "1/200(A3)" -> must become "A3: 1/200"
    - "SCALE 1:100" -> must become "1/100"

    Rules to follow
    - The output must always be `<paper size>: <scale>`. Some drawings only show a scale
      with no paper size; in that case, infer whether it is most likely A1 or A3 based on
      the scale value. If no scale is present at all, return 'NaN'. If nothing usable is
      present, also return 'NaN'.
    - If multiple candidates are found, prefer A1 if present, otherwise A3. Return exactly
      one scale value.
    - Return only the result, with no explanation.

    Previous response: "{scale_text}"

    Please reformat it according to the rules above. This is the final pass, so follow the
    rules carefully and output only the corrected final answer.
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

def process_segment(segment_image):
    """Use GPT to extract the drawing scale from one image segment."""
    base64_image = encode_image_object(segment_image)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Extract the drawing scale information from this image. "
                                "The output must always be `<paper size>: <scale>`. Some drawings only "
                                "show a scale with no paper size; in that case, infer whether it is most "
                                "likely A1 or A3 based on the scale value. If no scale is present, return "
                                "'NaN'. If nothing usable is present, also return 'NaN'. "
                                "Examples: 'A1: 1/40', 'A3: 1/80', '1/100'. "
                                "If multiple candidates are found, prefer A1 if present, otherwise A3. "
                                "Return exactly one scale value. "
                                "Return only the result, with no explanation."
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"}
                    }
                ],
            }
        ],
    )

    return response.choices[0].message.content.strip()

def validate_scale_info(scale_text):
    """Check whether the model's response matches the expected format."""
    pattern = r'^(A[1-4]):\s*\d{1,4}\s*[/:]\s*\d{1,4}$'
    return bool(re.match(pattern, scale_text))

def split_and_process_image(image_path, step=1000):
    """Tile the image into step x step segments and extract the scale from each."""
    image = Image.open(image_path)
    width, height = image.size
    results = []

    for y in range(0, height, step):
        for x in range(0, width, step):
            segment = image.crop((x, y, min(x + step, width), min(y + step, height)))
            gpt_result = process_segment(segment)

            # Re-ask the model if its response doesn't match the expected format
            if not validate_scale_info(gpt_result):
                gpt_result = request_gpt_fix(gpt_result)

            if gpt_result != "NaN":
                results.append(((x, y), gpt_result))

    return results

def process_single_image(image_path, step=1000):
    """
    Process a single image and return (has_scale, scale_info).
    """
    segments_results = split_and_process_image(image_path, step)

    if segments_results:
        scale_info = segments_results[0][1]  # take the first valid result
        has_scale = True
    else:
        scale_info = "NaN"
        has_scale = False

    return has_scale, scale_info

# Batch-process the whole folder
input_dir = "Drawings/FP/Images"   # input image folder
output_dir = "Drawings/Scale"      # output JSON folder
os.makedirs(output_dir, exist_ok=True)

valid_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

image_paths = [
    os.path.join(input_dir, fname)
    for fname in os.listdir(input_dir)
    if os.path.splitext(fname)[1].lower() in valid_exts
]

if not image_paths:
    print(f"[WARNING] No images found in: {input_dir}")
else:
    print(f"\n=== Processing ({len(image_paths)}) files...")

for image_path in sorted(image_paths):
    try:
        has_scale, scale_info = process_single_image(image_path, step=1000)

        result_dict = {
            "image_name": os.path.basename(image_path),
            "has_scale": has_scale,
            "scale_info": scale_info
        }

        base = os.path.splitext(os.path.basename(image_path))[0]
        output_json_path = os.path.join(output_dir, f"{base}.json")

        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(result_dict, f, ensure_ascii=False, indent=4)

        print(f"[SAVED] {output_json_path}")
    except Exception as e:
        print(f"[ERROR] Error processing {image_path}: {e}")

print("\n[DONE] All done. Results are in:", output_dir)

#### 5.2. Dimensional calibration (update json file)

In [ ]:
import os
import json
import numpy as np
import re

# A-series paper sizes (mm)
paper_size_mm = {
    "A1": (594, 841),
    "A2": (420, 594),
    "A3": (297, 420),
    "A4": (210, 297)
}

def load_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(json_data, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=4)
    print(f"[SAVED] {output_path}")

def parse_scale_info(scale_info: str):
    if ':' in scale_info:
        parts = scale_info.split(':', 1)
        paper_part = parts[0].strip()
        ratio_part = parts[1].strip()
    else:
        paper_part = "A3"
        ratio_part = "1/100"
    nums = re.findall(r'\d+', ratio_part)
    if len(nums) >= 2:
        scale_factor = int(nums[-1])
    elif len(nums) == 1:
        scale_factor = int(nums[0])
    else:
        scale_factor = 100
    return paper_part, scale_factor

def process_json(detection_data, ocr_data):
    scale_info_str = ocr_data.get("scale_info", "A3: 1/100")
    paper_part, scale_factor = parse_scale_info(scale_info_str)

    if paper_part in paper_size_mm:
        paper_w_mm, paper_h_mm = paper_size_mm[paper_part]
    else:
        print(f"[WARN] Unknown paper '{paper_part}'. Using A3.")
        paper_w_mm, paper_h_mm = paper_size_mm["A3"]

    ocr_image_name = ocr_data.get("image_name")
    if not ocr_image_name:
        print("[ERROR] image_name missing in OCR JSON.")
        return detection_data

    matched_image_obj = None
    for img in detection_data.get("images", []):
        if img["file_name"] == ocr_image_name:
            matched_image_obj = img
            break

    if not matched_image_obj:
        print(f"[ERROR] No matching image for {ocr_image_name}")
        return detection_data

    width_px = matched_image_obj["width"]
    px_to_mm = (paper_w_mm / width_px) * scale_factor

    for ann in detection_data.get("annotations", []):
        if ann["image_id"] == matched_image_obj["id"]:

            if "original_center" in ann:
                ann["original_center"] = [v * px_to_mm for v in ann["original_center"]]

            if "transformed_center" in ann:
                ann["transformed_center"] = [v * px_to_mm for v in ann["transformed_center"]]

            if "bbox" in ann:
                ann["bbox"] = [v * px_to_mm for v in ann["bbox"]]

            if "start_point" in ann:
                ann["start_point"] = [v * px_to_mm for v in ann["start_point"]]

            if "end_point" in ann:
                ann["end_point"] = [v * px_to_mm for v in ann["end_point"]]

    matched_image_obj["paper_part"] = paper_part
    matched_image_obj["scale_factor"] = scale_factor
    matched_image_obj["conversion_factor"] = px_to_mm

    return detection_data

def calibrate_all_files():
    ocr_dir = "Drawings/Scale"
    detection_dir = "Drawings/PostProcessing/Skeleton8"
    output_dir = "Drawings/Calibrated"
    os.makedirs(output_dir, exist_ok=True)

    ocr_files = {f: os.path.join(ocr_dir, f) for f in os.listdir(ocr_dir) if f.endswith(".json")}
    detection_files = [f for f in os.listdir(detection_dir) if f.endswith(".json")]

    if not detection_files:
        print("[WARNING] No detection JSON files found.")
        return

    print(f"\n=== Scaling Start: {len(detection_files)} files ===")

    for det_file in detection_files:
        det_json_path = os.path.join(detection_dir, det_file)

        try:
            detection_data = load_json(det_json_path)
        except Exception:
            print(f"[ERROR] Cannot load detection JSON: {det_file}")
            continue

        # Match the detection JSON's image to its corresponding OCR/scale JSON
        img_name = detection_data["images"][0]["file_name"]
        ocr_json_name = os.path.splitext(img_name)[0] + ".json"

        if ocr_json_name not in ocr_files:
            print(f"[SKIP] No OCR result for: {img_name}")
            continue

        ocr_json_path = ocr_files[ocr_json_name]
        ocr_data = load_json(ocr_json_path)

        updated_data = process_json(detection_data, ocr_data)

        out_path = os.path.join(output_dir, det_file)
        save_json(updated_data, out_path)

    print("\n[DONE] All scaling completed. Output saved.")

calibrate_all_files()

In [ ]:
import os
import json
from glob import glob

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def finalize_calibrated_to_final(input_dir="Drawings/Calibrated", output_dir="Drawings/Final"):
    ensure_dir(output_dir)

    json_files = sorted(glob(os.path.join(input_dir, "*.json")))
    if not json_files:
        print(f"[WARNING] No JSON files found in: {input_dir}")
        return

    print("\n=== Calibrated -> Final export start ===")
    print(f"Input : {input_dir}")
    print(f"Output: {output_dir}")
    print(f"Total : {len(json_files)} files\n")

    for i, src in enumerate(json_files, 1):
        base = os.path.basename(src)
        dst = os.path.join(output_dir, base)

        try:
            with open(src, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Written back unchanged (pretty-printed only, no data/logic changes)
            with open(dst, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=4)

            print(f"[{i}/{len(json_files)}] Saved -> {dst}")
        except Exception as e:
            print(f"[{i}/{len(json_files)}] [ERROR] {src}: {e}")

    print("\n[DONE] All calibrated files exported to Final.")


finalize_calibrated_to_final()